**Last code edit:** 2026-08-16 14:56 (UTC+03:00)

In [761]:
import json
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def md(text):
    """Render professor-facing explanations as Markdown from executable cells."""
    display(Markdown(text))

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

In [762]:
md("""# BI130 Project - Module 0  
## Reproducible Data Preparation and Cleaning

**Dataset:** HetRec 2011 - Last.fm 2K  
**Purpose:** Produce one audited set of cleaned relational tables for the later graph, community, text-retrieval, and recommender notebooks.

This notebook follows four rules:

1. **Check first, change second:** we verify what the data looks like before touching it.
2. **Only merge what we reviewed ourselves:** no fuzzy matching, so nothing gets combined just because it looks similar.
3. **Keep the evidence:** every merge, removal and oddity is saved to its own file, so our choices can be checked.
4. **Clean here, model later:** this notebook prepares the data but builds no graphs, search engines or recommenders. Those belong to Modules 1 to 4.

The project brief assigns friendship and listening data to Modules 1-2, tag preprocessing to Module 3, and recommender modelling to Module 4. This notebook prepares all of those inputs while keeping the phases visibly separate.""")

# BI130 Project - Module 0  
## Reproducible Data Preparation and Cleaning

**Dataset:** HetRec 2011 - Last.fm 2K  
**Purpose:** Produce one audited set of cleaned relational tables for the later graph, community, text-retrieval, and recommender notebooks.

This notebook follows four rules:

1. **Check first, change second:** we verify what the data looks like before touching it.
2. **Only merge what we reviewed ourselves:** no fuzzy matching, so nothing gets combined just because it looks similar.
3. **Keep the evidence:** every merge, removal and oddity is saved to its own file, so our choices can be checked.
4. **Clean here, model later:** this notebook prepares the data but builds no graphs, search engines or recommenders. Those belong to Modules 1 to 4.

The project brief assigns friendship and listening data to Modules 1-2, tag preprocessing to Module 3, and recommender modelling to Module 4. This notebook prepares all of those inputs while keeping the phases visibly separate.

In [763]:
md("""## Chapter 1 - Reproducible project paths

The project location is the notebook's current working directory:

```python
Path.cwd()
```

The raw `.dat` files are read from its `data` subfolder, and every cleaned output is written to `clean`. This keeps the notebook portable when the whole project folder is moved to another computer.""")

## Chapter 1 - Reproducible project paths

The project location is the notebook's current working directory:

```python
Path.cwd()
```

The raw `.dat` files are read from its `data` subfolder, and every cleaned output is written to `clean`. This keeps the notebook portable when the whole project folder is moved to another computer.

In [764]:
from pathlib import Path
script_dir = Path.cwd()
PROJECT_DIR = script_dir
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")

PROJECT_DIR: /Users/anton.gladyshev/git/music-recommendation-system
DATA_DIR:    /Users/anton.gladyshev/git/music-recommendation-system/data
OUTPUT_DIR:  /Users/anton.gladyshev/git/music-recommendation-system/clean


In [765]:
md("""### Download the source dataset

If the `data` folder is missing, this step downloads the official HetRec 2011 Last.fm archive from GroupLens, extracts its files into a temporary folder, and renames that folder to `data`. If `data` already exists, nothing is downloaded or changed.""")

import urllib.request
import zipfile

DATASET_URL = "https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip"
ARCHIVE_PATH = PROJECT_DIR / "hetrec2011-lastfm-2k.zip"
EXTRACTED_DIR = PROJECT_DIR / "hetrec2011-lastfm-2k"
REQUIRED_DATA_FILES = {
    "artists.dat",
    "tags.dat",
    "user_artists.dat",
    "user_friends.dat",
    "user_taggedartists.dat",
    "user_taggedartists-timestamps.dat",
}

if DATA_DIR.exists() and not DATA_DIR.is_dir():
    raise NotADirectoryError(f"Expected a directory at {DATA_DIR}")

if DATA_DIR.is_dir():
    print(f"Dataset folder already exists: {DATA_DIR}")
else:
    if not ARCHIVE_PATH.exists():
        print(f"Downloading {DATASET_URL}")
        urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)

    if EXTRACTED_DIR.exists():
        raise FileExistsError(
            f"Remove the incomplete extraction folder before retrying: {EXTRACTED_DIR}"
        )

    print(f"Extracting {ARCHIVE_PATH.name}")
    EXTRACTED_DIR.mkdir()
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(EXTRACTED_DIR)

    missing_files = sorted(
        filename
        for filename in REQUIRED_DATA_FILES
        if not (EXTRACTED_DIR / filename).is_file()
    )
    if missing_files:
        raise FileNotFoundError(f"Files missing from extracted archive: {missing_files}")

    EXTRACTED_DIR.rename(DATA_DIR)
    print(f"Dataset is ready: {DATA_DIR}")

### Download the source dataset

If the `data` folder is missing, this step downloads the official HetRec 2011 Last.fm archive from GroupLens, extracts its files into a temporary folder, and renames that folder to `data`. If `data` already exists, nothing is downloaded or changed.

Dataset folder already exists: /Users/anton.gladyshev/git/music-recommendation-system/data


In [766]:
md("""## Chapter 2 - Scope and module boundaries

### Phase A: shared structural cleaning

This phase prepares:

- the artist lookup;
- the user-artist listening table;
- the directed and undirected friendship tables;
- the user universe;
- a traceable artist-ID mapping.

These outputs feed Modules 1, 2, and 4.

### Phase B: tag preprocessing

This phase prepares:

- validated timestamped tag events;
- orphan-removal evidence;
- normalized tag keys and display forms;
- the approved 140-pair singular/plural review;
- conservative spelling-equivalence mappings;
- one vote per user, artist, and canonical tag;
- semantic and rare-tag eligibility fields.

These outputs feed Module 3 and are later reused by Modules 2 and 4.

### Explicitly deferred

The following approved modelling choices are **recorded but not executed here**:

- `1 + log(distinct_user_count)` for Module 3 artist-tag strength;
- `np.log1p(weight)` for Module 4 user profiles and implicit-feedback confidence;
- a per-user 80/20 split with `RANDOM_STATE = 42` for Module 4 evaluation.

When this notebook says "approved", it means a decision we made ourselves and wrote down while checking the data, so anyone reading later can see what was decided and why.""")

## Chapter 2 - Scope and module boundaries

### Phase A: shared structural cleaning

This phase prepares:

- the artist lookup;
- the user-artist listening table;
- the directed and undirected friendship tables;
- the user universe;
- a traceable artist-ID mapping.

These outputs feed Modules 1, 2, and 4.

### Phase B: tag preprocessing

This phase prepares:

- validated timestamped tag events;
- orphan-removal evidence;
- normalized tag keys and display forms;
- the approved 140-pair singular/plural review;
- conservative spelling-equivalence mappings;
- one vote per user, artist, and canonical tag;
- semantic and rare-tag eligibility fields.

These outputs feed Module 3 and are later reused by Modules 2 and 4.

### Explicitly deferred

The following approved modelling choices are **recorded but not executed here**:

- `1 + log(distinct_user_count)` for Module 3 artist-tag strength;
- `np.log1p(weight)` for Module 4 user profiles and implicit-feedback confidence;
- a per-user 80/20 split with `RANDOM_STATE = 42` for Module 4 evaluation.

When this notebook says "approved", it means a decision we made ourselves and wrote down while checking the data, so anyone reading later can see what was decided and why.

In [767]:
md("""## Chapter 3 - Source-file inventory

The official README describes six tab-separated source files. Before reading any data, the notebook verifies that all six files exist. A missing file stops the pipeline immediately because partial cleaning would create misleading outputs.""")

## Chapter 3 - Source-file inventory

The official README describes six tab-separated source files. Before reading any data, the notebook verifies that all six files exist. A missing file stops the pipeline immediately because partial cleaning would create misleading outputs.

In [768]:
SOURCE_FILES = {
    "artists": DATA_DIR / "artists.dat",
    "tags": DATA_DIR / "tags.dat",
    "user_artists": DATA_DIR / "user_artists.dat",
    "user_friends": DATA_DIR / "user_friends.dat",
    "user_taggedartists": DATA_DIR / "user_taggedartists.dat",
    "user_taggedartists_timestamps": DATA_DIR / "user_taggedartists-timestamps.dat",
}

source_inventory = pd.DataFrame([
    {
        "dataset": name,
        "filename": path.name,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan,
    }
    for name, path in SOURCE_FILES.items()
])

display(source_inventory)

missing_files = source_inventory.loc[~source_inventory["exists"], "filename"].tolist()
assert not missing_files, f"Missing source files: {missing_files}"

,dataset,filename,exists,size_bytes
0,artists,artists.dat,True,1925879
1,tags,tags.dat,True,234468
2,user_artists,user_artists.dat,True,1296455
3,user_friends,user_friends.dat,True,251565
4,user_taggedartists,user_taggedartists.dat,True,4366696
5,user_taggedartists_timestamps,user_taggedartists-timestamps.dat,True,5249917


In [769]:
md("""## Chapter 4 - Load the raw tables

The files are tab-separated. `tags.dat` uses Latin-1, while the other files load correctly as UTF-8 or numeric tabular data.

Explicit source names are retained (`*_raw`) so later cells cannot accidentally overwrite the original inputs.

We did not guess the file encodings. Opening `tags.dat` as UTF-8 makes Python raise an error, so it has to be read as latin-1. The catch is that latin-1 accepts any byte without complaining, so the check below makes sure nothing was quietly garbled: we searched for tags showing the classic broken-accent pattern and found none. This matters because Module 3 builds its search vocabulary from these tag texts and Module 2 uses them to describe communities. If the tags were corrupted here, everything built on top of them would be wrong.""")

## Chapter 4 - Load the raw tables

The files are tab-separated. `tags.dat` uses Latin-1, while the other files load correctly as UTF-8 or numeric tabular data.

Explicit source names are retained (`*_raw`) so later cells cannot accidentally overwrite the original inputs.

We did not guess the file encodings. Opening `tags.dat` as UTF-8 makes Python raise an error, so it has to be read as latin-1. The catch is that latin-1 accepts any byte without complaining, so the check below makes sure nothing was quietly garbled: we searched for tags showing the classic broken-accent pattern and found none. This matters because Module 3 builds its search vocabulary from these tag texts and Module 2 uses them to describe communities. If the tags were corrupted here, everything built on top of them would be wrong.

In [770]:
artists_raw = pd.read_csv(SOURCE_FILES["artists"], sep="\t", encoding="utf-8")
tags_raw = pd.read_csv(SOURCE_FILES["tags"], sep="\t", encoding="latin-1")
user_artists_raw = pd.read_csv(SOURCE_FILES["user_artists"], sep="\t")
user_friends_raw = pd.read_csv(SOURCE_FILES["user_friends"], sep="\t")
tag_dates_raw = pd.read_csv(SOURCE_FILES["user_taggedartists"], sep="\t")
tag_timestamps_raw = pd.read_csv(SOURCE_FILES["user_taggedartists_timestamps"], sep="\t")

RAW_TABLES = {
    "artists": artists_raw,
    "tags": tags_raw,
    "user_artists": user_artists_raw,
    "user_friends": user_friends_raw,
    "user_taggedartists": tag_dates_raw,
    "user_taggedartists_timestamps": tag_timestamps_raw,
}

raw_shapes = pd.DataFrame([
    {"dataset": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in RAW_TABLES.items()
])
display(raw_shapes)

,dataset,rows,columns
0,artists,17632,4
1,tags,11946,2
2,user_artists,92834,3
3,user_friends,25434,2
4,user_taggedartists,186479,6
5,user_taggedartists_timestamps,186479,4


In [771]:
# How the encodings were chosen and verified.
try:
    pd.read_csv(SOURCE_FILES["tags"], sep="\t", encoding="utf-8")
    tags_reads_as_utf8 = True
except UnicodeDecodeError:
    tags_reads_as_utf8 = False

assert not tags_reads_as_utf8

def double_encoding_repair(value):
    """Return the repaired string if value looks double-encoded, else None."""
    try:
        repaired = str(value).encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return None
    return repaired if repaired != str(value) else None

tag_mojibake_candidates = tags_raw["tagValue"].map(double_encoding_repair).dropna()
assert tag_mojibake_candidates.empty
print("tags.dat cannot be read as UTF-8 and contains no double-encoded values under latin-1.")

tags.dat cannot be read as UTF-8 and contains no double-encoded values under latin-1.


In [772]:
md("""## Chapter 5 - Validate published row counts and schemas

We compare our row counts against the numbers published in the official README. If they do not match, the notebook stops immediately, because that would mean we downloaded the wrong version or a broken file. Every later module quotes these counts, so they need to be right from the very start.""")

## Chapter 5 - Validate published row counts and schemas

We compare our row counts against the numbers published in the official README. If they do not match, the notebook stops immediately, because that would mean we downloaded the wrong version or a broken file. Every later module quotes these counts, so they need to be right from the very start.

In [773]:
EXPECTED_ROWS = {
    "artists": 17_632,
    "tags": 11_946,
    "user_artists": 92_834,
    "user_friends": 25_434,
    "user_taggedartists": 186_479,
    "user_taggedartists_timestamps": 186_479,
}

EXPECTED_COLUMNS = {
    "artists": ["id", "name", "url", "pictureURL"],
    "tags": ["tagID", "tagValue"],
    "user_artists": ["userID", "artistID", "weight"],
    "user_friends": ["userID", "friendID"],
    "user_taggedartists": ["userID", "artistID", "tagID", "day", "month", "year"],
    "user_taggedartists_timestamps": ["userID", "artistID", "tagID", "timestamp"],
}

schema_checks = []
for name, frame in RAW_TABLES.items():
    row_match = len(frame) == EXPECTED_ROWS[name]
    column_match = frame.columns.tolist() == EXPECTED_COLUMNS[name]
    schema_checks.append({
        "dataset": name,
        "expected_rows": EXPECTED_ROWS[name],
        "actual_rows": len(frame),
        "row_count_matches": row_match,
        "columns_match": column_match,
    })
    assert row_match, f"{name}: expected {EXPECTED_ROWS[name]} rows, found {len(frame)}"
    assert column_match, (
        f"{name}: expected columns {EXPECTED_COLUMNS[name]}, "
        f"found {frame.columns.tolist()}"
    )

schema_checks = pd.DataFrame(schema_checks)
display(schema_checks)

,dataset,expected_rows,actual_rows,row_count_matches,columns_match
0,artists,17632,17632,True,True
1,tags,11946,11946,True,True
2,user_artists,92834,92834,True,True
3,user_friends,25434,25434,True,True
4,user_taggedartists,186479,186479,True,True
5,user_taggedartists_timestamps,186479,186479,True,True


In [774]:
md("""## Chapter 6 - Structural data-quality audit

Not every gap in the data is a problem. A missing artist picture link hurts nothing, because no module uses pictures. A duplicate row or a broken ID would poison everything built on top. This chapter checks for the serious kind: duplicate rows, missing IDs, negative play counts, and references to artists or tags that do not exist. The graphs in Modules 1 and 2 and the recommendation tables in Module 4 all assume IDs are unique and links are real, so this is where that assumption gets earned.""")

## Chapter 6 - Structural data-quality audit

Not every gap in the data is a problem. A missing artist picture link hurts nothing, because no module uses pictures. A duplicate row or a broken ID would poison everything built on top. This chapter checks for the serious kind: duplicate rows, missing IDs, negative play counts, and references to artists or tags that do not exist. The graphs in Modules 1 and 2 and the recommendation tables in Module 4 all assume IDs are unique and links are real, so this is where that assumption gets earned.

In [775]:
required_key_columns = {
    "artists": ["id", "name"],
    "tags": ["tagID", "tagValue"],
    "user_artists": ["userID", "artistID", "weight"],
    "user_friends": ["userID", "friendID"],
    "user_taggedartists": ["userID", "artistID", "tagID", "day", "month", "year"],
    "user_taggedartists_timestamps": ["userID", "artistID", "tagID", "timestamp"],
}

structural_audit_rows = []
for name, frame in RAW_TABLES.items():
    structural_audit_rows.append({
        "dataset": name,
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "missing_required_values": int(frame[required_key_columns[name]].isna().sum().sum()),
    })

structural_audit = pd.DataFrame(structural_audit_rows)

assert structural_audit["exact_duplicate_rows"].sum() == 0
assert structural_audit["missing_required_values"].sum() == 0
assert artists_raw["id"].is_unique
assert tags_raw["tagID"].is_unique
assert not user_artists_raw.duplicated(["userID", "artistID"]).any()
assert not user_friends_raw.duplicated(["userID", "friendID"]).any()
assert (user_artists_raw["weight"] > 0).all()

known_artist_ids_raw = set(artists_raw["id"])
known_tag_ids_raw = set(tags_raw["tagID"])

listen_unknown_artists = ~user_artists_raw["artistID"].isin(known_artist_ids_raw)
tag_unknown_tags = ~tag_dates_raw["tagID"].isin(known_tag_ids_raw)
tag_unknown_artists = ~tag_dates_raw["artistID"].isin(known_artist_ids_raw)

assert listen_unknown_artists.sum() == 0
assert tag_unknown_tags.sum() == 0

structural_audit_extra = pd.DataFrame([
    {"check": "artists.id is unique", "value": artists_raw["id"].is_unique},
    {"check": "tags.tagID is unique", "value": tags_raw["tagID"].is_unique},
    {"check": "user-artist pairs are unique", "value": not user_artists_raw.duplicated(["userID", "artistID"]).any()},
    {"check": "all listening weights are positive", "value": (user_artists_raw["weight"] > 0).all()},
    {"check": "listening rows with unknown artists", "value": int(listen_unknown_artists.sum())},
    {"check": "tag rows with unknown tag IDs", "value": int(tag_unknown_tags.sum())},
    {"check": "tag rows with unknown artist IDs", "value": int(tag_unknown_artists.sum())},
    {"check": "unknown artist IDs in tag rows", "value": int(tag_dates_raw.loc[tag_unknown_artists, "artistID"].nunique())},
    {"check": "missing pictureURL values (documented only)", "value": int(artists_raw["pictureURL"].isna().sum())},
])

display(structural_audit)
display(structural_audit_extra)

,dataset,exact_duplicate_rows,missing_required_values
0,artists,0,0
1,tags,0,0
2,user_artists,0,0
3,user_friends,0,0
4,user_taggedartists,0,0
5,user_taggedartists_timestamps,0,0


,check,value
0,artists.id is unique,True
1,tags.tagID is unique,True
2,user-artist pairs are unique,True
3,all listening weights are positive,True
4,listening rows with unknown artists,0
5,tag rows with unknown tag IDs,0
6,tag rows with unknown artist IDs,1538
7,unknown artist IDs in tag rows,390
8,missing pictureURL values (documented only),444


In [776]:
md("""## Chapter 7 - Construct the user universe

The dataset has no file listing the users, so we build one ourselves from everyone who appears anywhere. The three yes/no columns record where each user shows up. This helps later: a user can sit in the Module 1 friendship graph but have no listening history for Module 4, and these flags let us explain that instead of being surprised by it.""")

## Chapter 7 - Construct the user universe

The dataset has no file listing the users, so we build one ourselves from everyone who appears anywhere. The three yes/no columns record where each user shows up. This helps later: a user can sit in the Module 1 friendship graph but have no listening history for Module 4, and these flags let us explain that instead of being surprised by it.

In [777]:
all_user_ids = sorted(
    set(user_artists_raw["userID"])
    | set(user_friends_raw["userID"])
    | set(user_friends_raw["friendID"])
    | set(tag_dates_raw["userID"])
)

users_clean = pd.DataFrame({"userID": all_user_ids})
users_clean["has_listening_data"] = users_clean["userID"].isin(user_artists_raw["userID"])
users_clean["appears_in_friendship_graph"] = (
    users_clean["userID"].isin(user_friends_raw["userID"])
    | users_clean["userID"].isin(user_friends_raw["friendID"])
)
users_clean["has_tag_assignments"] = users_clean["userID"].isin(tag_dates_raw["userID"])

assert len(users_clean) == 1_892
display(users_clean.head())
display(users_clean[["has_listening_data", "appears_in_friendship_graph", "has_tag_assignments"]].sum().to_frame("users"))

,userID,has_listening_data,appears_in_friendship_graph,has_tag_assignments
0,2,True,True,True
1,3,True,True,True
2,4,True,True,True
3,5,True,True,True
4,6,True,True,True


,users
has_listening_data,1892
appears_in_friendship_graph,1892
has_tag_assignments,1892


In [778]:
md("""# Phase A - Shared structural cleaning

## Chapter 8 - Detect exact artist-name duplicates conservatively

Duplicate artists are detected using only three harmless steps: trim spaces at the ends, squash repeated spaces in the middle, and ignore upper versus lower case. We deliberately do not remove accents or punctuation and do not use fuzzy matching, because names that merely look similar could be different artists. This matters for Module 1: if the same artist exists under two IDs, the artist graph treats them as two separate nodes and their listeners get split between them, which would distort both prestige and popularity.""")

# Phase A - Shared structural cleaning

## Chapter 8 - Detect exact artist-name duplicates conservatively

Duplicate artists are detected using only three harmless steps: trim spaces at the ends, squash repeated spaces in the middle, and ignore upper versus lower case. We deliberately do not remove accents or punctuation and do not use fuzzy matching, because names that merely look similar could be different artists. This matters for Module 1: if the same artist exists under two IDs, the artist graph treats them as two separate nodes and their listeners get split between them, which would distort both prestige and popularity.

In [779]:
def artist_duplicate_key(value):
    value = re.sub(r"\s+", " ", str(value).strip())
    return value.casefold()

artist_candidates = artists_raw.copy()
artist_candidates["duplicate_key"] = artist_candidates["name"].map(artist_duplicate_key)

duplicate_artist_groups = (
    artist_candidates[
        artist_candidates.duplicated("duplicate_key", keep=False)
    ]
    .groupby("duplicate_key", as_index=False)
    .agg(
        source_ids=("id", lambda values: " | ".join(map(str, sorted(values)))),
        source_names=("name", lambda values: " | ".join(values)),
        group_size=("id", "size"),
    )
    .sort_values("duplicate_key")
    .reset_index(drop=True)
)

assert len(duplicate_artist_groups) == 12
assert int((duplicate_artist_groups["group_size"] - 1).sum()) == 13
display(duplicate_artist_groups)

,duplicate_key,source_ids,source_names,group_size
0,eldad lidor,13305 | 13306,Eldad Lidor | Eldad Lidor,2
1,michel teló,7091 | 12654,Michel Teló | MICHEL TELÓ,2
2,бригадный подряд,2736 | 10497,Бригадный Подряд | БРИГАДНЫЙ ПОДРЯД,2
3,гости из будущего,3857 | 4302 | 13360,Гости Из Будущего | Гости из будущего | Гости из Будущего,3
4,ежи и петруччо,7362 | 7363,Ежи И Петруччо | Ежи и Петруччо,2
5,записки неизвестного,8611 | 14179,Записки Неизвестного | Записки неизвестного,2
6,король и шут,10504 | 11170,Король И Шут | Король и Шут,2
7,настя,3849 | 3851,НАСТЯ | Настя,2
8,пси(х)ея,3835 | 3853,Пси(Х)еЯ | Пси(Х)ея,2
9,розовые очки от ferre,8631 | 16196,Розовые очки от ferre | Розовые Очки От Ferre,2


In [780]:
md("""## Chapter 9 - Select the surviving artist ID

For each approved duplicate group, the surviving ID is selected by:

1. the largest number of distinct listeners;
2. the largest total listening weight if tied;
3. the smallest artist ID if still tied.

Distinct listeners are placed before total weight so one extreme listener cannot determine identity. Every source ID, including survivors, is retained in the mapping table.""")

## Chapter 9 - Select the surviving artist ID

For each approved duplicate group, the surviving ID is selected by:

1. the largest number of distinct listeners;
2. the largest total listening weight if tied;
3. the smallest artist ID if still tied.

Distinct listeners are placed before total weight so one extreme listener cannot determine identity. Every source ID, including survivors, is retained in the mapping table.

In [781]:
artist_usage = (
    user_artists_raw.groupby("artistID", as_index=False)
    .agg(
        distinct_listeners=("userID", "nunique"),
        total_listens=("weight", "sum"),
    )
)

artist_scored = (
    artist_candidates
    .merge(artist_usage, left_on="id", right_on="artistID", how="left")
    .drop(columns="artistID")
)

artist_scored[["distinct_listeners", "total_listens"]] = (
    artist_scored[["distinct_listeners", "total_listens"]]
    .fillna(0)
    .astype("int64")
)

artist_id_map = {int(artist_id): int(artist_id) for artist_id in artists_raw["id"]}
artist_merge_records = []

duplicate_rows = artist_scored[
    artist_scored.duplicated("duplicate_key", keep=False)
]

for duplicate_key, group in duplicate_rows.groupby("duplicate_key"):
    ranked = group.sort_values(
        ["distinct_listeners", "total_listens", "id"],
        ascending=[False, False, True],
    )
    survivor = ranked.iloc[0]

    for _, row in ranked.iterrows():
        source_id = int(row["id"])
        final_id = int(survivor["id"])
        artist_id_map[source_id] = final_id

        artist_merge_records.append({
            "duplicate_key": duplicate_key,
            "source_artistID": source_id,
            "source_name": row["name"],
            "source_distinct_listeners": int(row["distinct_listeners"]),
            "source_total_listens": int(row["total_listens"]),
            "final_artistID": final_id,
            "survivor_source_name": survivor["name"],
            "is_survivor": source_id == final_id,
        })

artist_merge_audit = pd.DataFrame(artist_merge_records)
absorbed_artist_ids = {
    source_id for source_id, final_id in artist_id_map.items()
    if source_id != final_id
}

assert len(absorbed_artist_ids) == 13
display(artist_merge_audit)

,duplicate_key,source_artistID,source_name,source_distinct_listeners,source_total_listens,final_artistID,survivor_source_name,is_survivor
0,eldad lidor,13305,Eldad Lidor,1,83,13305,Eldad Lidor,True
1,eldad lidor,13306,Eldad Lidor,1,72,13305,Eldad Lidor,False
2,michel teló,7091,Michel Teló,4,23479,7091,Michel Teló,True
3,michel teló,12654,MICHEL TELÓ,1,346,7091,Michel Teló,False
4,бригадный подряд,2736,Бригадный Подряд,1,5081,2736,Бригадный Подряд,True
5,бригадный подряд,10497,БРИГАДНЫЙ ПОДРЯД,1,138,2736,Бригадный Подряд,False
6,гости из будущего,3857,Гости Из Будущего,1,606,3857,Гости Из Будущего,True
7,гости из будущего,13360,Гости из Будущего,1,249,3857,Гости Из Будущего,False
8,гости из будущего,4302,Гости из будущего,1,163,3857,Гости Из Будущего,False
9,ежи и петруччо,7362,Ежи И Петруччо,1,268,7362,Ежи И Петруччо,True


In [782]:
md("""## Chapter 10 - Repair the confirmed encoding defect and create canonical names

Only one reviewed encoding repair is applied:

```text
BallakÃ© Sissoko -> Ballaké Sissoko
```

No broad automatic encoding repair is used because similar byte patterns may be legitimate names.

The canonical artist label then:

- begins with the surviving ID's repaired name;
- removes combining accents;
- applies casefolding;
- replaces punctuation with spaces;
- preserves letters and numbers from all writing systems;
- collapses repeated spaces.

This canonical label is for consistent presentation. It is **not** used to discover further duplicate artists, because the more aggressive formatting can create legitimate name collisions.

Before fixing any artist name, we tested all 17,632 of them with a simple reversibility trick: if a name was saved with broken encoding, undoing the mistake produces a sensible name, and if the name is genuine, the trick fails. Only one name passed: `BallakÃ© Sissoko`, which becomes `Ballaké Sissoko`. Its own Last.fm web address contains the same broken characters, which shows the mistake came from the source data, not from us. The similar-looking `OSKÃO` failed the test, and its web address shows that character really is part of the name, so we left it alone. Getting this right means the artist tables in Modules 1, 2 and 4 show real artists instead of garbled text.""")

## Chapter 10 - Repair the confirmed encoding defect and create canonical names

Only one reviewed encoding repair is applied:

```text
BallakÃ© Sissoko -> Ballaké Sissoko
```

No broad automatic encoding repair is used because similar byte patterns may be legitimate names.

The canonical artist label then:

- begins with the surviving ID's repaired name;
- removes combining accents;
- applies casefolding;
- replaces punctuation with spaces;
- preserves letters and numbers from all writing systems;
- collapses repeated spaces.

This canonical label is for consistent presentation. It is **not** used to discover further duplicate artists, because the more aggressive formatting can create legitimate name collisions.

Before fixing any artist name, we tested all 17,632 of them with a simple reversibility trick: if a name was saved with broken encoding, undoing the mistake produces a sensible name, and if the name is genuine, the trick fails. Only one name passed: `BallakÃ© Sissoko`, which becomes `Ballaké Sissoko`. Its own Last.fm web address contains the same broken characters, which shows the mistake came from the source data, not from us. The similar-looking `OSKÃO` failed the test, and its web address shows that character really is part of the name, so we left it alone. Getting this right means the artist tables in Modules 1, 2 and 4 show real artists instead of garbled text.

In [783]:
artist_mojibake_candidates = (
    artists_raw.assign(repaired=artists_raw["name"].map(double_encoding_repair))
    .dropna(subset=["repaired"])
)

assert artist_mojibake_candidates["name"].tolist() == ["BallakÃ© Sissoko"]
assert artist_mojibake_candidates["repaired"].tolist() == ["Ballaké Sissoko"]
assert artist_mojibake_candidates["url"].str.contains("%C3%83%C2%A9").all()

oskao_url = artists_raw.loc[artists_raw["name"] == "OSKÃO", "url"].iloc[0]
assert "OSK%C3%83O" in oskao_url

display(artist_mojibake_candidates[["id", "name", "repaired", "url"]])

,id,name,repaired,url
11293,11681,BallakÃ© Sissoko,Ballaké Sissoko,http://www.last.fm/music/Ballak%C3%83%C2%A9+Sissoko


In [784]:
ENCODING_FIXES = {
    "BallakÃ© Sissoko": "Ballaké Sissoko",
}

def normalize_artist_name(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(
        character
        for character in value
        if not unicodedata.combining(character)
    )
    value = value.casefold()
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)
    value = re.sub(r"_+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()

artist_id_mapping = artists_raw[["id", "name"]].copy()
artist_id_mapping = artist_id_mapping.rename(
    columns={"id": "source_artistID", "name": "source_name"}
)
artist_id_mapping["final_artistID"] = (
    artist_id_mapping["source_artistID"].map(artist_id_map).astype("int64")
)
artist_id_mapping["is_absorbed"] = (
    artist_id_mapping["source_artistID"] != artist_id_mapping["final_artistID"]
)

survivor_lookup = (
    artists_raw.set_index("id")["name"]
    .rename("survivor_source_name")
)
artist_id_mapping["survivor_source_name"] = (
    artist_id_mapping["final_artistID"].map(survivor_lookup)
)

survivor_ids = sorted(set(artist_id_map.values()))
artists_clean = (
    artists_raw[artists_raw["id"].isin(survivor_ids)]
    .rename(columns={"id": "artistID"})
    .copy()
)

artists_clean["source_name_raw"] = artists_clean["name"]
artists_clean["source_name_repaired"] = (
    artists_clean["source_name_raw"].replace(ENCODING_FIXES)
)
artists_clean["canonical_name"] = (
    artists_clean["source_name_repaired"].map(normalize_artist_name)
)

artists_clean = artists_clean[
    [
        "artistID",
        "canonical_name",
        "source_name_repaired",
        "source_name_raw",
        "url",
        "pictureURL",
    ]
].sort_values("artistID").reset_index(drop=True)

assert len(artists_clean) == 17_619
assert artists_clean["artistID"].is_unique
assert artists_clean["source_name_raw"].eq("BallakÃ© Sissoko").sum() == 1
assert artists_clean["source_name_repaired"].eq("Ballaké Sissoko").sum() == 1

print(f"Artists before cleaning: {len(artists_raw):,}")
print(f"Absorbed duplicate IDs: {len(absorbed_artist_ids):,}")
print(f"Artists after cleaning:  {len(artists_clean):,}")
print(
    "Canonical-name collisions retained as separate IDs:",
    int(artists_clean["canonical_name"].duplicated(keep=False).sum()),
)
display(artists_clean.head())

Artists before cleaning: 17,632
Absorbed duplicate IDs: 13
Artists after cleaning:  17,619
Canonical-name collisions retained as separate IDs: 139


,artistID,canonical_name,source_name_repaired,source_name_raw,url,pictureURL
0,1,malice mizer,MALICE MIZER,MALICE MIZER,http://www.last.fm/music/MALICE+MIZER,http://userserve-ak.last.fm/serve/252/10808.jpg
1,2,diary of dreams,Diary of Dreams,Diary of Dreams,http://www.last.fm/music/Diary+of+Dreams,http://userserve-ak.last.fm/serve/252/3052066.jpg
2,3,carpathian forest,Carpathian Forest,Carpathian Forest,http://www.last.fm/music/Carpathian+Forest,http://userserve-ak.last.fm/serve/252/40222717.jpg
3,4,moi dix mois,Moi dix Mois,Moi dix Mois,http://www.last.fm/music/Moi+dix+Mois,http://userserve-ak.last.fm/serve/252/54697835.png
4,5,bella morte,Bella Morte,Bella Morte,http://www.last.fm/music/Bella+Morte,http://userserve-ak.last.fm/serve/252/14789013.jpg


In [785]:
md("""## Chapter 11 - Remap and aggregate listening records

The artist merges are now applied to the listening data. When two IDs turn out to be the same artist, that artist's play counts are added together rather than lost. Play counts are the raw material for the artist graph in Module 1 and both recommenders in Module 4, so the checks confirm the total number of plays is exactly the same before and after. Not a single play may appear or disappear during cleaning.""")

## Chapter 11 - Remap and aggregate listening records

The artist merges are now applied to the listening data. When two IDs turn out to be the same artist, that artist's play counts are added together rather than lost. Play counts are the raw material for the artist graph in Module 1 and both recommenders in Module 4, so the checks confirm the total number of plays is exactly the same before and after. Not a single play may appear or disappear during cleaning.

In [786]:
listens_work = user_artists_raw.copy()
listens_work["source_artistID"] = listens_work["artistID"]
listens_work["artistID"] = listens_work["artistID"].map(artist_id_map)

assert listens_work["artistID"].notna().all()
listens_work["artistID"] = listens_work["artistID"].astype("int64")

total_listening_weight_before = int(listens_work["weight"].sum())

user_artists_clean = (
    listens_work.groupby(["userID", "artistID"], as_index=False)
    .agg(
        weight=("weight", "sum"),
        source_row_count=("source_artistID", "size"),
    )
    .sort_values(["userID", "artistID"])
    .reset_index(drop=True)
)

total_listening_weight_after = int(user_artists_clean["weight"].sum())

assert total_listening_weight_before == total_listening_weight_after
assert not user_artists_clean.duplicated(["userID", "artistID"]).any()
assert (user_artists_clean["weight"] > 0).all()
assert set(user_artists_clean["artistID"]).issubset(set(artists_clean["artistID"]))

listening_reconciliation = pd.DataFrame([
    {"measure": "raw user-artist rows", "value": len(user_artists_raw)},
    {"measure": "clean user-artist rows", "value": len(user_artists_clean)},
    {"measure": "rows consolidated after artist merge", "value": len(user_artists_raw) - len(user_artists_clean)},
    {"measure": "total weight before", "value": total_listening_weight_before},
    {"measure": "total weight after", "value": total_listening_weight_after},
])
display(listening_reconciliation)

,measure,value
0,raw user-artist rows,92834
1,clean user-artist rows,92829
2,rows consolidated after artist merge,5
3,total weight before,69183975
4,total weight after,69183975


In [787]:
md("""## Chapter 12 - Validate and canonicalize friendships

The friendship file stores every friendship twice, once in each direction. We keep that original form for traceability, and also build a second table where each friendship appears exactly once. Modules 1 and 2 treat friendship as mutual, so their graph needs one edge per pair, not two. The checks confirm nobody is friends with themselves, every friendship really does appear in both directions, and 25,434 rows collapse to exactly the 12,717 friendships the README promises.""")

## Chapter 12 - Validate and canonicalize friendships

The friendship file stores every friendship twice, once in each direction. We keep that original form for traceability, and also build a second table where each friendship appears exactly once. Modules 1 and 2 treat friendship as mutual, so their graph needs one edge per pair, not two. The checks confirm nobody is friends with themselves, every friendship really does appear in both directions, and 25,434 rows collapse to exactly the 12,717 friendships the README promises.

In [788]:
user_friends_directed_clean = (
    user_friends_raw[["userID", "friendID"]]
    .sort_values(["userID", "friendID"])
    .reset_index(drop=True)
)

directed_pairs = set(
    map(tuple, user_friends_directed_clean[["userID", "friendID"]].to_numpy())
)

self_loop_count = int(
    (user_friends_directed_clean["userID"] == user_friends_directed_clean["friendID"]).sum()
)
missing_reverse_count = sum(
    (friend_id, user_id) not in directed_pairs
    for user_id, friend_id in directed_pairs
)

assert self_loop_count == 0
assert missing_reverse_count == 0
assert not user_friends_directed_clean.duplicated(["userID", "friendID"]).any()

user_friends_undirected_clean = pd.DataFrame({
    "userID_a": np.minimum(
        user_friends_directed_clean["userID"],
        user_friends_directed_clean["friendID"],
    ),
    "userID_b": np.maximum(
        user_friends_directed_clean["userID"],
        user_friends_directed_clean["friendID"],
    ),
}).drop_duplicates().sort_values(["userID_a", "userID_b"]).reset_index(drop=True)

assert len(user_friends_directed_clean) == 25_434
assert len(user_friends_undirected_clean) == 12_717
assert not user_friends_undirected_clean.duplicated(["userID_a", "userID_b"]).any()

friendship_reconciliation = pd.DataFrame([
    {"measure": "directed rows", "value": len(user_friends_directed_clean)},
    {"measure": "undirected edges", "value": len(user_friends_undirected_clean)},
    {"measure": "self-loops", "value": self_loop_count},
    {"measure": "directed edges missing reverse", "value": missing_reverse_count},
])
display(friendship_reconciliation)

,measure,value
0,directed rows,25434
1,undirected edges,12717
2,self-loops,0
3,directed edges missing reverse,0


In [789]:
md("""# Phase B - Tag preprocessing for Module 3

## Chapter 13 - Reconcile calendar dates with Unix timestamps

The two tag-assignment files describe the same 186,479 events:

- one contains day, month, and year;
- the other contains Unix time in milliseconds.

The event keys must align exactly. The Unix timestamps are interpreted as UTC and converted to `Europe/Madrid`, which reproduces every source calendar date.

Five pre-2005 dates are implausible for Last.fm. They are retained and flagged rather than deleted because the tag relationships remain usable and timestamps are not used as semantic weights.

The two tag files describe the same moments in time, one as a calendar date and one as a computer timestamp, so they should agree. To compare them we had to pick a timezone. Plain UTC left about 183,000 dates that did not line up. The dataset was created at a university in Madrid, so we tried Spanish local time instead, and every single date matched. This proves the two files are consistent, so later modules can trust either one.""")

# Phase B - Tag preprocessing for Module 3

## Chapter 13 - Reconcile calendar dates with Unix timestamps

The two tag-assignment files describe the same 186,479 events:

- one contains day, month, and year;
- the other contains Unix time in milliseconds.

The event keys must align exactly. The Unix timestamps are interpreted as UTC and converted to `Europe/Madrid`, which reproduces every source calendar date.

Five pre-2005 dates are implausible for Last.fm. They are retained and flagged rather than deleted because the tag relationships remain usable and timestamps are not used as semantic weights.

The two tag files describe the same moments in time, one as a calendar date and one as a computer timestamp, so they should agree. To compare them we had to pick a timezone. Plain UTC left about 183,000 dates that did not line up. The dataset was created at a university in Madrid, so we tried Spanish local time instead, and every single date matched. This proves the two files are consistent, so later modules can trust either one.

In [790]:
event_keys = ["userID", "artistID", "tagID"]

assert tag_dates_raw[event_keys].equals(tag_timestamps_raw[event_keys])

tag_events = tag_dates_raw.copy()
tag_events["timestamp_ms"] = tag_timestamps_raw["timestamp"].astype("int64")
tag_events["timestamp_utc_dt"] = pd.to_datetime(
    tag_events["timestamp_ms"],
    unit="ms",
    utc=True,
)
tag_events["timestamp_madrid_dt"] = (
    tag_events["timestamp_utc_dt"].dt.tz_convert("Europe/Madrid")
)

calendar_matches_timestamp = (
    (tag_events["timestamp_madrid_dt"].dt.day == tag_events["day"])
    & (tag_events["timestamp_madrid_dt"].dt.month == tag_events["month"])
    & (tag_events["timestamp_madrid_dt"].dt.year == tag_events["year"])
)

assert calendar_matches_timestamp.all()

tag_events["is_timestamp_anomaly"] = tag_events["year"] < 2005

assert int(tag_events["is_timestamp_anomaly"].sum()) == 5

timestamp_validation = pd.DataFrame([
    {"measure": "date-file rows", "value": len(tag_dates_raw)},
    {"measure": "timestamp-file rows", "value": len(tag_timestamps_raw)},
    {"measure": "calendar mismatches after Europe/Madrid conversion", "value": int((~calendar_matches_timestamp).sum())},
    {"measure": "pre-2005 timestamp anomalies retained", "value": int(tag_events["is_timestamp_anomaly"].sum())},
])
display(timestamp_validation)
display(tag_events.loc[tag_events["is_timestamp_anomaly"], event_keys + ["day", "month", "year", "timestamp_ms"]])

,measure,value
0,date-file rows,186479
1,timestamp-file rows,186479
2,calendar mismatches after Europe/Madrid conversion,0
3,pre-2005 timestamp anomalies retained,5


,userID,artistID,tagID,day,month,year,timestamp_ms
4144,43,1395,39,1,6,1956,-428720400000
13129,133,2984,1474,1,3,1957,-405133200000
137311,1604,1583,103,1,9,1956,-420771600000
137314,1604,1583,10021,1,9,1956,-420771600000
170911,1929,250,311,1,5,1979,294357600000


In [791]:
md("""## Chapter 14 - Remap tag artists and isolate orphan assignments

1,538 tag assignments point to artist IDs that do not exist in the artist file. We cannot tell which artist they meant, so they cannot join any graph, artist profile or recommendation. We remove them from the analysis tables but save every removed row to its own file as evidence. This is the only place in the whole notebook where data is dropped, and nothing disappears without a trace.""")

## Chapter 14 - Remap tag artists and isolate orphan assignments

1,538 tag assignments point to artist IDs that do not exist in the artist file. We cannot tell which artist they meant, so they cannot join any graph, artist profile or recommendation. We remove them from the analysis tables but save every removed row to its own file as evidence. This is the only place in the whole notebook where data is dropped, and nothing disappears without a trace.

In [792]:
tag_events["source_artistID"] = tag_events["artistID"]
tag_events["artistID"] = tag_events["artistID"].map(artist_id_map)

orphan_mask = tag_events["artistID"].isna()

orphan_assignments_removed = tag_events.loc[orphan_mask].copy()
valid_tag_events = tag_events.loc[~orphan_mask].copy()
valid_tag_events["artistID"] = valid_tag_events["artistID"].astype("int64")

orphan_assignments_removed = (
    orphan_assignments_removed
    .merge(
        tags_raw.rename(columns={"tagValue": "raw_tag"}),
        on="tagID",
        how="left",
        validate="many_to_one",
    )
)

assert len(orphan_assignments_removed) == 1_538
assert orphan_assignments_removed["source_artistID"].nunique() == 390
assert len(valid_tag_events) == 184_941
assert set(valid_tag_events["artistID"]).issubset(set(artists_clean["artistID"]))

orphan_reconciliation = pd.DataFrame([
    {"measure": "raw tag events", "value": len(tag_events)},
    {"measure": "valid tag events retained", "value": len(valid_tag_events)},
    {"measure": "orphan rows removed from analysis", "value": len(orphan_assignments_removed)},
    {"measure": "unknown source artist IDs", "value": orphan_assignments_removed["source_artistID"].nunique()},
])
display(orphan_reconciliation)

,measure,value
0,raw tag events,186479
1,valid tag events retained,184941
2,orphan rows removed from analysis,1538
3,unknown source artist IDs,390


In [793]:
md("""## Chapter 15 - Basic tag normalization

For every tag we make a tidied-up version: lowercase, extra spaces removed, and spaces, hyphens, underscores and slashes treated as the same separator. That way `female-vocals`, `female_vocals` and `female vocals` count as one tag. We deliberately stop there: `synthpop` is not automatically assumed to equal `synth pop`, because gluing words together can combine genuinely different concepts. Each tag keeps two forms, a readable one for display and a technical key for matching. Module 3's search engine and Module 2's community descriptions are both built on these keys, so mistakes here would surface everywhere.""")

## Chapter 15 - Basic tag normalization

For every tag we make a tidied-up version: lowercase, extra spaces removed, and spaces, hyphens, underscores and slashes treated as the same separator. That way `female-vocals`, `female_vocals` and `female vocals` count as one tag. We deliberately stop there: `synthpop` is not automatically assumed to equal `synth pop`, because gluing words together can combine genuinely different concepts. Each tag keeps two forms, a readable one for display and a technical key for matching. Module 3's search engine and Module 2's community descriptions are both built on these keys, so mistakes here would surface everywhere.

In [794]:
def clean_tag_text(value):
    value = unicodedata.normalize("NFKC", str(value))
    value = re.sub(r"\s+", " ", value.strip())
    return value.casefold()

def make_tag_key(value):
    value = clean_tag_text(value)
    value = re.sub(r"[\s\-_/]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

valid_tag_usage = (
    valid_tag_events.groupby("tagID", as_index=False)
    .agg(
        raw_distinct_users=("userID", "nunique"),
        raw_distinct_artists=("artistID", "nunique"),
        raw_assignments=("userID", "size"),
    )
)

tag_mapping = (
    tags_raw.rename(columns={"tagValue": "raw_tag"})
    .merge(valid_tag_usage, on="tagID", how="left")
)

tag_mapping[
    ["raw_distinct_users", "raw_distinct_artists", "raw_assignments"]
] = (
    tag_mapping[
        ["raw_distinct_users", "raw_distinct_artists", "raw_assignments"]
    ]
    .fillna(0)
    .astype("int64")
)

tag_mapping["display_normalized"] = tag_mapping["raw_tag"].map(clean_tag_text)
tag_mapping["tag_key"] = tag_mapping["raw_tag"].map(make_tag_key)

base_display_by_key = (
    tag_mapping.sort_values(
        ["raw_distinct_users", "raw_assignments", "display_normalized"],
        ascending=[False, False, True],
    )
    .groupby("tag_key")["display_normalized"]
    .first()
)

tag_mapping["base_display_tag"] = tag_mapping["tag_key"].map(base_display_by_key)

print(f"Raw tag IDs:             {len(tag_mapping):,}")
print(f"Basic normalized keys:   {tag_mapping['tag_key'].nunique():,}")
display(tag_mapping.head())

Raw tag IDs:             11,946
Basic normalized keys:   11,801


,tagID,raw_tag,raw_distinct_users,raw_distinct_artists,raw_assignments,display_normalized,tag_key,base_display_tag
0,1,metal,237,638,1722,metal,metal,metal
1,2,alternative metal,62,101,212,alternative metal,alternative metal,alternative metal
2,3,goth rock,5,20,22,goth rock,goth rock,goth rock
3,4,black metal,55,126,299,black metal,black metal,black metal
4,5,death metal,96,216,579,death metal,death metal,death metal


In [795]:
md("""## Chapter 16 - Apply the approved 140-pair singular/plural review

A simple final-`s` rule produces false matches such as:

- `blue` and `blues`;
- `Wale` and `Wales`;
- `prince`, `princes`, and `princess`;
- `Philippine` and `Philippines`.

For that reason, 140 candidates were manually reviewed. The approved decisions are embedded below so the notebook remains self-contained and reproducible.

Review outcome:

- 118 candidate pairs merged;
- 22 candidate pairs kept separate;
- 73 review pairs flagged for interpretation.

A flag does not automatically mean deletion. It identifies concepts requiring careful downstream use.

The 140 candidate pairs were found automatically, by checking for every tag whether the same tag plus a letter `s` also exists (like `ballad` and `ballads`). That rule is good at finding possible duplicates but terrible at judging them, since `blue` and `blues` mean completely different things. So we went through all 140 by hand. This stops the Module 3 search vocabulary from mixing up different concepts while still merging tags that obviously belong together.""")

## Chapter 16 - Apply the approved 140-pair singular/plural review

A simple final-`s` rule produces false matches such as:

- `blue` and `blues`;
- `Wale` and `Wales`;
- `prince`, `princes`, and `princess`;
- `Philippine` and `Philippines`.

For that reason, 140 candidates were manually reviewed. The approved decisions are embedded below so the notebook remains self-contained and reproducible.

Review outcome:

- 118 candidate pairs merged;
- 22 candidate pairs kept separate;
- 73 review pairs flagged for interpretation.

A flag does not automatically mean deletion. It identifies concepts requiring careful downstream use.

The 140 candidate pairs were found automatically, by checking for every tag whether the same tag plus a letter `s` also exists (like `ballad` and `ballads`). That rule is good at finding possible duplicates but terrible at judging them, since `blue` and `blues` mean completely different things. So we went through all 140 by hand. This stops the Module 3 search vocabulary from mixing up different concepts while still merging tags that obviously belong together.

In [796]:
APPROVED_PLURAL_REVIEW = pd.DataFrame(json.loads(r"""[
  {
    "review_no": 1,
    "singular_tag": "female vocalist",
    "plural_tag": "female vocalists",
    "decision": "Merge",
    "merge_into": "female vocalist",
    "reviewer_notes": null
  },
  {
    "review_no": 2,
    "singular_tag": "singer-songwriter",
    "plural_tag": "singer-songwriters",
    "decision": "Merge",
    "merge_into": "singer-songwriter",
    "reviewer_notes": null
  },
  {
    "review_no": 3,
    "singular_tag": "instrumental",
    "plural_tag": "instrumentals",
    "decision": "Merge",
    "merge_into": "instrumental",
    "reviewer_notes": null
  },
  {
    "review_no": 4,
    "singular_tag": "soul",
    "plural_tag": "souls",
    "decision": "Merge",
    "merge_into": "soul",
    "reviewer_notes": null
  },
  {
    "review_no": 5,
    "singular_tag": "male vocalist",
    "plural_tag": "male vocalists",
    "decision": "Merge",
    "merge_into": "male vocalist",
    "reviewer_notes": null
  },
  {
    "review_no": 6,
    "singular_tag": "soundtrack",
    "plural_tag": "soundtracks",
    "decision": "Merge",
    "merge_into": "soundtrack",
    "reviewer_notes": null
  },
  {
    "review_no": 7,
    "singular_tag": "cover",
    "plural_tag": "covers",
    "decision": "Merge",
    "merge_into": "cover",
    "reviewer_notes": null
  },
  {
    "review_no": 8,
    "singular_tag": "american",
    "plural_tag": "americans",
    "decision": "Merge",
    "merge_into": "american",
    "reviewer_notes": null
  },
  {
    "review_no": 9,
    "singular_tag": "favorite",
    "plural_tag": "favorites",
    "decision": "Merge",
    "merge_into": "favorite",
    "reviewer_notes": "Flagged: personal/subjective preference."
  },
  {
    "review_no": 10,
    "singular_tag": "blue",
    "plural_tag": "blues",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate: `blue` may describe mood; `blues` is a genre."
  },
  {
    "review_no": 11,
    "singular_tag": "ballad",
    "plural_tag": "ballads",
    "decision": "Merge",
    "merge_into": "ballad",
    "reviewer_notes": null
  },
  {
    "review_no": 12,
    "singular_tag": "oldie",
    "plural_tag": "oldies",
    "decision": "Merge",
    "merge_into": "oldies",
    "reviewer_notes": null
  },
  {
    "review_no": 13,
    "singular_tag": "guilty pleasure",
    "plural_tag": "guilty pleasures",
    "decision": "Merge",
    "merge_into": "guilty pleasure",
    "reviewer_notes": "Flagged: personal/subjective preference."
  },
  {
    "review_no": 14,
    "singular_tag": "guitar",
    "plural_tag": "guitars",
    "decision": "Merge",
    "merge_into": "guitar",
    "reviewer_notes": null
  },
  {
    "review_no": 15,
    "singular_tag": "favorite song",
    "plural_tag": "favorite songs",
    "decision": "Merge",
    "merge_into": "favorite song",
    "reviewer_notes": "Flagged: personal/subjective preference."
  },
  {
    "review_no": 16,
    "singular_tag": "legend",
    "plural_tag": "legends",
    "decision": "Merge",
    "merge_into": "legend",
    "reviewer_notes": "Flagged: subjective/evaluative label."
  },
  {
    "review_no": 17,
    "singular_tag": "chill",
    "plural_tag": "chills",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate: `chill` and `chills` represent different emotional concepts."
  },
  {
    "review_no": 18,
    "singular_tag": "favourite",
    "plural_tag": "favourites",
    "decision": "Merge",
    "merge_into": "favourite",
    "reviewer_notes": null
  },
  {
    "review_no": 19,
    "singular_tag": "love song",
    "plural_tag": "love songs",
    "decision": "Merge",
    "merge_into": "love song",
    "reviewer_notes": null
  },
  {
    "review_no": 20,
    "singular_tag": "new romantic",
    "plural_tag": "new romantics",
    "decision": "Merge",
    "merge_into": "new romantic",
    "reviewer_notes": null
  },
  {
    "review_no": 21,
    "singular_tag": "diva",
    "plural_tag": "divas",
    "decision": "Merge",
    "merge_into": "diva",
    "reviewer_notes": null
  },
  {
    "review_no": 22,
    "singular_tag": "fave",
    "plural_tag": "faves",
    "decision": "Merge",
    "merge_into": "fave",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 23,
    "singular_tag": "vocal",
    "plural_tag": "vocals",
    "decision": "Merge",
    "merge_into": "vocal",
    "reviewer_notes": null
  },
  {
    "review_no": 24,
    "singular_tag": "my favorite",
    "plural_tag": "my favorites",
    "decision": "Merge",
    "merge_into": "my favorite",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 25,
    "singular_tag": "duet",
    "plural_tag": "duets",
    "decision": "Merge",
    "merge_into": "duet",
    "reviewer_notes": null
  },
  {
    "review_no": 26,
    "singular_tag": "american idol",
    "plural_tag": "american idols",
    "decision": "Merge",
    "merge_into": "american idol",
    "reviewer_notes": null
  },
  {
    "review_no": 27,
    "singular_tag": "favorite artist",
    "plural_tag": "favorite artists",
    "decision": "Merge",
    "merge_into": "favorite artist",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 28,
    "singular_tag": "female vocal",
    "plural_tag": "female vocals",
    "decision": "Merge",
    "merge_into": "female vocals",
    "reviewer_notes": null
  },
  {
    "review_no": 29,
    "singular_tag": "great song",
    "plural_tag": "great songs",
    "decision": "Merge",
    "merge_into": "great song",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 30,
    "singular_tag": "fav song",
    "plural_tag": "fav songs",
    "decision": "Merge",
    "merge_into": "fav song",
    "reviewer_notes": "Flagged: personal/subjective preference."
  },
  {
    "review_no": 31,
    "singular_tag": "composer",
    "plural_tag": "composers",
    "decision": "Merge",
    "merge_into": "composer",
    "reviewer_notes": null
  },
  {
    "review_no": 32,
    "singular_tag": "idolo",
    "plural_tag": "idolos",
    "decision": "Merge",
    "merge_into": "idolo",
    "reviewer_notes": "Flagged: personal/subjective category."
  },
  {
    "review_no": 33,
    "singular_tag": "b-side",
    "plural_tag": "b-sides",
    "decision": "Merge",
    "merge_into": "b-side",
    "reviewer_notes": null
  },
  {
    "review_no": 34,
    "singular_tag": "favorite band",
    "plural_tag": "favorite bands",
    "decision": "Merge",
    "merge_into": "favorite band",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 35,
    "singular_tag": "summer song",
    "plural_tag": "summer songs",
    "decision": "Merge",
    "merge_into": "summer song",
    "reviewer_notes": null
  },
  {
    "review_no": 36,
    "singular_tag": "masterpiece",
    "plural_tag": "masterpieces",
    "decision": "Merge",
    "merge_into": "masterpiece",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 37,
    "singular_tag": "female voice",
    "plural_tag": "female voices",
    "decision": "Merge",
    "merge_into": "female voice",
    "reviewer_notes": null
  },
  {
    "review_no": 38,
    "singular_tag": "musical",
    "plural_tag": "musicals",
    "decision": "Merge",
    "merge_into": "musical",
    "reviewer_notes": null
  },
  {
    "review_no": 39,
    "singular_tag": "boyband",
    "plural_tag": "boybands",
    "decision": "Merge",
    "merge_into": "boyband",
    "reviewer_notes": null
  },
  {
    "review_no": 40,
    "singular_tag": "saxophone",
    "plural_tag": "saxophones",
    "decision": "Merge",
    "merge_into": "saxophone",
    "reviewer_notes": null
  },
  {
    "review_no": 41,
    "singular_tag": "beat",
    "plural_tag": "beats",
    "decision": "Merge",
    "merge_into": "beat",
    "reviewer_notes": null
  },
  {
    "review_no": 42,
    "singular_tag": "my favorite song",
    "plural_tag": "myfavoritesongs",
    "decision": "Merge",
    "merge_into": "my favorite song",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 43,
    "singular_tag": "sexy song",
    "plural_tag": "sexy songs",
    "decision": "Merge",
    "merge_into": "sexy song",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 44,
    "singular_tag": "color",
    "plural_tag": "colors",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate: insufficient evidence that the forms have the same intended meaning."
  },
  {
    "review_no": 45,
    "singular_tag": "dream",
    "plural_tag": "dreams",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate: related concepts, but not clearly equivalent in the assignments."
  },
  {
    "review_no": 46,
    "singular_tag": "songwriter",
    "plural_tag": "songwriters",
    "decision": "Merge",
    "merge_into": "songwriter",
    "reviewer_notes": null
  },
  {
    "review_no": 47,
    "singular_tag": "male vocal",
    "plural_tag": "male vocals",
    "decision": "Merge",
    "merge_into": "male vocals",
    "reviewer_notes": null
  },
  {
    "review_no": 48,
    "singular_tag": "new discovery",
    "plural_tag": "new discoverys",
    "decision": "Merge",
    "merge_into": "new discovery",
    "reviewer_notes": null
  },
  {
    "review_no": 49,
    "singular_tag": "girl group",
    "plural_tag": "girl groups",
    "decision": "Merge",
    "merge_into": "girl group",
    "reviewer_notes": null
  },
  {
    "review_no": 50,
    "singular_tag": "queen",
    "plural_tag": "queens",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. `queen` is ambiguous/subjective; `queens` may be geographic."
  },
  {
    "review_no": 51,
    "singular_tag": "animal",
    "plural_tag": "animals",
    "decision": "Merge",
    "merge_into": "animals",
    "reviewer_notes": "Flagged: thematic or potentially personal category."
  },
  {
    "review_no": 52,
    "singular_tag": "bad day",
    "plural_tag": "bad days",
    "decision": "Merge",
    "merge_into": "bad day",
    "reviewer_notes": "Flagged: mood/listening-context."
  },
  {
    "review_no": 53,
    "singular_tag": "drug",
    "plural_tag": "drugs",
    "decision": "Merge",
    "merge_into": "drugs",
    "reviewer_notes": "Flagged: thematic/ambiguous."
  },
  {
    "review_no": 54,
    "singular_tag": "demo",
    "plural_tag": "demos",
    "decision": "Merge",
    "merge_into": "demo",
    "reviewer_notes": null
  },
  {
    "review_no": 55,
    "singular_tag": "violin",
    "plural_tag": "violins",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate: `violins` may be an artist-name tag rather than instrumentation."
  },
  {
    "review_no": 56,
    "singular_tag": "setenta",
    "plural_tag": "setentas",
    "decision": "Merge",
    "merge_into": "setenta",
    "reviewer_notes": null
  },
  {
    "review_no": 57,
    "singular_tag": "idol",
    "plural_tag": "idols",
    "decision": "Merge",
    "merge_into": "idol",
    "reviewer_notes": "Flagged: personal/ambiguous."
  },
  {
    "review_no": 58,
    "singular_tag": "rock ballad",
    "plural_tag": "rock ballads",
    "decision": "Merge",
    "merge_into": "rock ballad",
    "reviewer_notes": null
  },
  {
    "review_no": 59,
    "singular_tag": "rainy day",
    "plural_tag": "rainy days",
    "decision": "Merge",
    "merge_into": "rainy day",
    "reviewer_notes": "Flagged: mood/listening-context."
  },
  {
    "review_no": 60,
    "singular_tag": "wale",
    "plural_tag": "wales",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate: `Wale` is an artist name; `Wales` is geographic."
  },
  {
    "review_no": 61,
    "singular_tag": "drum",
    "plural_tag": "drums",
    "decision": "Merge",
    "merge_into": "drums",
    "reviewer_notes": null
  },
  {
    "review_no": 62,
    "singular_tag": "flute",
    "plural_tag": "flutes",
    "decision": "Merge",
    "merge_into": "flute",
    "reviewer_notes": null
  },
  {
    "review_no": 63,
    "singular_tag": "great cover",
    "plural_tag": "great covers",
    "decision": "Merge",
    "merge_into": "great cover",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 64,
    "singular_tag": "great voice",
    "plural_tag": "great voices",
    "decision": "Merge",
    "merge_into": "great voice",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 65,
    "singular_tag": "original",
    "plural_tag": "originals",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate: `original` and `originals` may represent different categories."
  },
  {
    "review_no": 66,
    "singular_tag": "vocalist",
    "plural_tag": "vocalists",
    "decision": "Merge",
    "merge_into": "vocalist",
    "reviewer_notes": null
  },
  {
    "review_no": 67,
    "singular_tag": "big band",
    "plural_tag": "big bands",
    "decision": "Merge",
    "merge_into": "big band",
    "reviewer_notes": null
  },
  {
    "review_no": 68,
    "singular_tag": "gay icon",
    "plural_tag": "gay icons",
    "decision": "Merge",
    "merge_into": "gay icon",
    "reviewer_notes": "Flagged: cultural/contextual."
  },
  {
    "review_no": 69,
    "singular_tag": "all time favorite",
    "plural_tag": "all time favorites",
    "decision": "Merge",
    "merge_into": "all time favorite",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 70,
    "singular_tag": "beautiful voice",
    "plural_tag": "beautiful voices",
    "decision": "Merge",
    "merge_into": "beautiful voice",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 71,
    "singular_tag": "favorite album",
    "plural_tag": "favorite albums",
    "decision": "Merge",
    "merge_into": "favorite album",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 72,
    "singular_tag": "mashup",
    "plural_tag": "mash ups",
    "decision": "Merge",
    "merge_into": "mashup",
    "reviewer_notes": null
  },
  {
    "review_no": 73,
    "singular_tag": "song",
    "plural_tag": "songs",
    "decision": "Merge",
    "merge_into": "song",
    "reviewer_notes": "Flagged: generic/low-information."
  },
  {
    "review_no": 74,
    "singular_tag": "girl",
    "plural_tag": "girls",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: ambiguous gender/group classification."
  },
  {
    "review_no": 75,
    "singular_tag": "great riff",
    "plural_tag": "great riffs",
    "decision": "Merge",
    "merge_into": "great riff",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 76,
    "singular_tag": "heart",
    "plural_tag": "hearts",
    "decision": "Merge",
    "merge_into": "heart",
    "reviewer_notes": "Flagged: thematic/ambiguous."
  },
  {
    "review_no": 77,
    "singular_tag": "linda",
    "plural_tag": "lindas",
    "decision": "Merge",
    "merge_into": "linda",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 78,
    "singular_tag": "band",
    "plural_tag": "bands",
    "decision": "Merge",
    "merge_into": "band",
    "reviewer_notes": "Flagged: generic/low-information."
  },
  {
    "review_no": 79,
    "singular_tag": "girl band",
    "plural_tag": "girl bands",
    "decision": "Merge",
    "merge_into": "girl band",
    "reviewer_notes": null
  },
  {
    "review_no": 80,
    "singular_tag": "question",
    "plural_tag": "questions",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: ambiguous/private organisational label."
  },
  {
    "review_no": 81,
    "singular_tag": "string",
    "plural_tag": "strings",
    "decision": "Merge",
    "merge_into": "strings",
    "reviewer_notes": null
  },
  {
    "review_no": 82,
    "singular_tag": "field recording",
    "plural_tag": "field recordings",
    "decision": "Merge",
    "merge_into": "field recording",
    "reviewer_notes": null
  },
  {
    "review_no": 83,
    "singular_tag": "guitar god",
    "plural_tag": "guitar gods",
    "decision": "Merge",
    "merge_into": "guitar god",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 84,
    "singular_tag": "johnny",
    "plural_tag": "johnnys",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: `johnnys` may be agency/contextual; `johnny` is unclear."
  },
  {
    "review_no": 85,
    "singular_tag": "soundscape",
    "plural_tag": "soundscapes",
    "decision": "Merge",
    "merge_into": "soundscape",
    "reviewer_notes": null
  },
  {
    "review_no": 86,
    "singular_tag": "light",
    "plural_tag": "lights",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: ambiguous meanings."
  },
  {
    "review_no": 87,
    "singular_tag": "my fav",
    "plural_tag": "my favs",
    "decision": "Merge",
    "merge_into": "my fav",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 88,
    "singular_tag": "outsider",
    "plural_tag": "outsiders",
    "decision": "Merge",
    "merge_into": "outsiders",
    "reviewer_notes": "Flagged: cultural/ambiguous."
  },
  {
    "review_no": 89,
    "singular_tag": "featuring",
    "plural_tag": "featurings",
    "decision": "Merge",
    "merge_into": "featuring",
    "reviewer_notes": null
  },
  {
    "review_no": 90,
    "singular_tag": "game soundtrack",
    "plural_tag": "game soundtracks",
    "decision": "Merge",
    "merge_into": "game soundtrack",
    "reviewer_notes": null
  },
  {
    "review_no": 91,
    "singular_tag": "perfect song",
    "plural_tag": "perfect songs",
    "decision": "Merge",
    "merge_into": "perfect song",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 92,
    "singular_tag": "power ballad",
    "plural_tag": "power ballads",
    "decision": "Merge",
    "merge_into": "power ballad",
    "reviewer_notes": null
  },
  {
    "review_no": 93,
    "singular_tag": "spoken word",
    "plural_tag": "spoken words",
    "decision": "Merge",
    "merge_into": "spoken word",
    "reviewer_notes": null
  },
  {
    "review_no": 94,
    "singular_tag": "jam band",
    "plural_tag": "jam bands",
    "decision": "Merge",
    "merge_into": "jam band",
    "reviewer_notes": null
  },
  {
    "review_no": 95,
    "singular_tag": "jihad song",
    "plural_tag": "jihad songs",
    "decision": "Merge",
    "merge_into": "jihad song",
    "reviewer_notes": "Flagged: sensitive/thematic; retain for analysis with careful interpretation."
  },
  {
    "review_no": 96,
    "singular_tag": "king",
    "plural_tag": "kings",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: `king` is subjective/honorific; `kings` is unclear."
  },
  {
    "review_no": 97,
    "singular_tag": "my favourite",
    "plural_tag": "my favourites",
    "decision": "Merge",
    "merge_into": "my favourite",
    "reviewer_notes": "Flagged: personal preference."
  },
  {
    "review_no": 98,
    "singular_tag": "pioneer",
    "plural_tag": "pioneers",
    "decision": "Merge",
    "merge_into": "pioneer",
    "reviewer_notes": "Flagged: evaluative/historical."
  },
  {
    "review_no": 99,
    "singular_tag": "awesome song",
    "plural_tag": "awesome songs",
    "decision": "Merge",
    "merge_into": "awesome song",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 100,
    "singular_tag": "boysband",
    "plural_tag": "boysbands",
    "decision": "Merge",
    "merge_into": "boysband",
    "reviewer_notes": null
  },
  {
    "review_no": 101,
    "singular_tag": "tune",
    "plural_tag": "tunes",
    "decision": "Merge",
    "merge_into": "tune",
    "reviewer_notes": "Flagged: generic/low-information."
  },
  {
    "review_no": 102,
    "singular_tag": "as tag",
    "plural_tag": "as tags",
    "decision": "Merge",
    "merge_into": "as tag",
    "reviewer_notes": "Flagged: meta/personal category."
  },
  {
    "review_no": 103,
    "singular_tag": "best song",
    "plural_tag": "best songs",
    "decision": "Merge",
    "merge_into": "best song",
    "reviewer_notes": "Flagged: subjective/personal."
  },
  {
    "review_no": 104,
    "singular_tag": "blog",
    "plural_tag": "blogs",
    "decision": "Merge",
    "merge_into": "blog",
    "reviewer_notes": "Flagged: contextual/meta."
  },
  {
    "review_no": 105,
    "singular_tag": "christmas song",
    "plural_tag": "christmas songs",
    "decision": "Merge",
    "merge_into": "christmas song",
    "reviewer_notes": null
  },
  {
    "review_no": 106,
    "singular_tag": "heavenly voice",
    "plural_tag": "heavenly voices",
    "decision": "Merge",
    "merge_into": "heavenly voice",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 107,
    "singular_tag": "prince",
    "plural_tag": "princes",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: `prince` may be an artist-name tag; `princes` an honorific."
  },
  {
    "review_no": 108,
    "singular_tag": "water",
    "plural_tag": "waters",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: ambiguous thematic/name reference."
  },
  {
    "review_no": 109,
    "singular_tag": "bone",
    "plural_tag": "bones",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: ambiguous/thematic."
  },
  {
    "review_no": 110,
    "singular_tag": "cover version",
    "plural_tag": "cover versions",
    "decision": "Merge",
    "merge_into": "cover version",
    "reviewer_notes": null
  },
  {
    "review_no": 111,
    "singular_tag": "g-e-n-i-o",
    "plural_tag": "g-e-n-i-o-s",
    "decision": "Merge",
    "merge_into": "g-e-n-i-o",
    "reviewer_notes": "Flagged: subjective/evaluative."
  },
  {
    "review_no": 112,
    "singular_tag": "movie",
    "plural_tag": "movies",
    "decision": "Merge",
    "merge_into": "movie",
    "reviewer_notes": "Flagged: film/contextual."
  },
  {
    "review_no": 113,
    "singular_tag": "princes",
    "plural_tag": "princess",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: not a singular/plural pair; both may be subjective honorifics."
  },
  {
    "review_no": 114,
    "singular_tag": "apologetic",
    "plural_tag": "apologetics",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: different meanings; `apologetics` is religious/contextual."
  },
  {
    "review_no": 115,
    "singular_tag": "cowboy",
    "plural_tag": "cowboys",
    "decision": "Merge",
    "merge_into": "cowboy",
    "reviewer_notes": "Flagged: thematic/cultural."
  },
  {
    "review_no": 116,
    "singular_tag": "great b-side",
    "plural_tag": "great b-sides",
    "decision": "Merge",
    "merge_into": "great b-side",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 117,
    "singular_tag": "great singer",
    "plural_tag": "great singers",
    "decision": "Merge",
    "merge_into": "great singer",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 118,
    "singular_tag": "hymn",
    "plural_tag": "hymns",
    "decision": "Merge",
    "merge_into": "hymns",
    "reviewer_notes": "Flagged: religious/thematic."
  },
  {
    "review_no": 119,
    "singular_tag": "lesbian",
    "plural_tag": "lesbians",
    "decision": "Merge",
    "merge_into": "lesbian",
    "reviewer_notes": "Flagged: identity-related/cultural/ambiguous; do not treat as verified identity."
  },
  {
    "review_no": 120,
    "singular_tag": "one hit wonder",
    "plural_tag": "one hit wonders",
    "decision": "Merge",
    "merge_into": "one hit wonder",
    "reviewer_notes": "Flagged: evaluative/contextual."
  },
  {
    "review_no": 121,
    "singular_tag": "thrasher",
    "plural_tag": "thrashers",
    "decision": "Merge",
    "merge_into": "thrasher",
    "reviewer_notes": null
  },
  {
    "review_no": 122,
    "singular_tag": "actor",
    "plural_tag": "actors",
    "decision": "Merge",
    "merge_into": "actor",
    "reviewer_notes": "Flagged: biographical/non-musical."
  },
  {
    "review_no": 123,
    "singular_tag": "children",
    "plural_tag": "childrens",
    "decision": "Merge",
    "merge_into": "children",
    "reviewer_notes": "Flagged: broad/contextual."
  },
  {
    "review_no": 124,
    "singular_tag": "dante",
    "plural_tag": "dantes",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: ambiguous/name-related."
  },
  {
    "review_no": 125,
    "singular_tag": "female singer",
    "plural_tag": "female singers",
    "decision": "Merge",
    "merge_into": "female singer",
    "reviewer_notes": null
  },
  {
    "review_no": 126,
    "singular_tag": "fractured vision",
    "plural_tag": "fractured visions",
    "decision": "Merge",
    "merge_into": "fractured visions",
    "reviewer_notes": "Flagged: artist-name/self-referential."
  },
  {
    "review_no": 127,
    "singular_tag": "french vocalist",
    "plural_tag": "french vocalists",
    "decision": "Merge",
    "merge_into": "french vocalist",
    "reviewer_notes": null
  },
  {
    "review_no": 128,
    "singular_tag": "great vocal",
    "plural_tag": "great vocals",
    "decision": "Merge",
    "merge_into": "great vocals",
    "reviewer_notes": "Flagged: subjective evaluation."
  },
  {
    "review_no": 129,
    "singular_tag": "latin singer",
    "plural_tag": "latin singers",
    "decision": "Merge",
    "merge_into": "latin singer",
    "reviewer_notes": null
  },
  {
    "review_no": 130,
    "singular_tag": "matt",
    "plural_tag": "matts",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: personal/name-related."
  },
  {
    "review_no": 131,
    "singular_tag": "movie soundtrack",
    "plural_tag": "movie soundtracks",
    "decision": "Merge",
    "merge_into": "movie soundtrack",
    "reviewer_notes": null
  },
  {
    "review_no": 132,
    "singular_tag": "piano song",
    "plural_tag": "piano songs",
    "decision": "Merge",
    "merge_into": "piano song",
    "reviewer_notes": null
  },
  {
    "review_no": 133,
    "singular_tag": "record",
    "plural_tag": "records",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate. Flagged: generic/ambiguous."
  },
  {
    "review_no": 134,
    "singular_tag": "track",
    "plural_tag": "tracks",
    "decision": "Merge",
    "merge_into": "track",
    "reviewer_notes": "Flagged: generic/low-information."
  },
  {
    "review_no": 135,
    "singular_tag": "vision",
    "plural_tag": "visions",
    "decision": "Merge",
    "merge_into": "visions",
    "reviewer_notes": "Flagged: artist-name/self-referential."
  },
  {
    "review_no": 136,
    "singular_tag": "audiobook",
    "plural_tag": "audiobooks",
    "decision": "Merge",
    "merge_into": "audiobook",
    "reviewer_notes": null
  },
  {
    "review_no": 137,
    "singular_tag": "human condition",
    "plural_tag": "human conditions",
    "decision": "Merge",
    "merge_into": "human condition",
    "reviewer_notes": "Flagged: thematic."
  },
  {
    "review_no": 138,
    "singular_tag": "tv show",
    "plural_tag": "tv shows",
    "decision": "Merge",
    "merge_into": "tv show",
    "reviewer_notes": "Flagged: media/contextual."
  },
  {
    "review_no": 139,
    "singular_tag": "debut album",
    "plural_tag": "debut albums",
    "decision": "Merge",
    "merge_into": "debut album",
    "reviewer_notes": null
  },
  {
    "review_no": 140,
    "singular_tag": "philippine",
    "plural_tag": "philippines",
    "decision": "Keep separate",
    "merge_into": null,
    "reviewer_notes": "Keep separate: `Philippine` is an adjective; `Philippines` is the country name."
  },
  {
    "review_no": 141,
    "review_author": "AGladyshev",
    "singular_tag": "00",
    "plural_tag": "00's",
    "decision": "Merge",
    "merge_into": "00",
    "reviewer_notes": null
  },
  {
      "review_no": 142,
      "review_author": "AGladyshev",
      "singular_tag": "00",
      "plural_tag": "00s",
      "decision": "Merge",
      "merge_into": "2000s",
      "reviewer_notes": null
  },
  {
      "review_no": 143,
      "review_author": "AGladyshev",
      "singular_tag": "1960s",
      "plural_tag": "1960's",
      "decision": "Merge",
      "merge_into": "1960s",
      "reviewer_notes": null
  },
  {
        "review_no": 144,
        "review_author": "AGladyshev",
        "singular_tag": "'80s",
        "plural_tag": "1980s",
        "decision": "Merge",
        "merge_into": "1980s",
        "reviewer_notes": null
    },
    {
        "review_no": 145,
        "review_author": "AGladyshev",
        "singular_tag": "1980s",
        "plural_tag": "1980 songs",
        "decision": "Merge",
        "merge_into": "1980s",
        "reviewer_notes": null
    },
    {
        "review_no": 146,
        "review_author": "AGladyshev",
        "singular_tag": "1980s",
        "plural_tag": "1980's",
        "decision": "Merge",
        "merge_into": "1980s",
        "reviewer_notes": null
    },
    {
        "review_no": 147,
        "review_author": "AGladyshev",
        "singular_tag": "1970s",
        "plural_tag": "1970's",
        "decision": "Merge",
        "merge_into": "1970s",
        "reviewer_notes": null
    },
    {
        "review_no": 148,
        "review_author": "AGladyshev",
        "singular_tag": "1981",
        "plural_tag": "1981 songs",
        "decision": "Merge",
        "merge_into": "1981",
        "reviewer_notes": null
    },
    {
        "review_no": 149,
        "review_author": "AGladyshev",
        "singular_tag": "1982",
        "plural_tag": "1982 songs",
        "decision": "Merge",
        "merge_into": "1982",
        "reviewer_notes": null
    },
    {
        "review_no": 150,
        "review_author": "AGladyshev",
        "singular_tag": "1983",
        "plural_tag": "1983 songs",
        "decision": "Merge",
        "merge_into": "1983",
        "reviewer_notes": null
    },
    {
        "review_no": 151,
        "review_author": "AGladyshev",
        "singular_tag": "1984",
        "plural_tag": "1984 songs",
        "decision": "Merge",
        "merge_into": "1984",
        "reviewer_notes": null
    },
    {
        "review_no": 152,
        "review_author": "AGladyshev",
        "singular_tag": "1985",
        "plural_tag": "1985 songs",
        "decision": "Merge",
        "merge_into": "1985",
        "reviewer_notes": null
    },
    {
        "review_no": 153,
        "review_author": "AGladyshev",
        "singular_tag": "1986",
        "plural_tag": "1986 songs",
        "decision": "Merge",
        "merge_into": "1986",
        "reviewer_notes": null
    },
    {
        "review_no": 154,
        "review_author": "AGladyshev",
        "singular_tag": "1990's",
        "plural_tag": "1990s",
        "decision": "Merge",
        "merge_into": "1990s",
        "reviewer_notes": null
    },
    {
        "review_no": 155,
        "review_author": "AGladyshev",
        "singular_tag": "2000's",
        "plural_tag": "2000s",
        "decision": "Merge",
        "merge_into": "2000s",
        "reviewer_notes": null
    },
     {
        "review_no": 156,
        "review_author": "AGladyshev",
        "singular_tag": "00s",
        "plural_tag": "2000s",
        "decision": "Merge",
        "merge_into": "2000s",
        "reviewer_notes": null
    },
    {
        "review_no": 157,
        "review_author": "AGladyshev",
        "singular_tag": "70's",
        "plural_tag": "1970s",
        "decision": "Merge",
        "merge_into": "1970s",
        "reviewer_notes": null
    },
    {
        "review_no": 158,
        "review_author": "AGladyshev",
        "singular_tag": "90s",
        "plural_tag": "1990s",
        "decision": "Merge",
        "merge_into": "1990s",
        "reviewer_notes": null
    },
    {
            "review_no": 159,
            "review_author": "AGladyshev",
            "singular_tag": "70",
            "plural_tag": "1970s",
            "decision": "Merge",
            "merge_into": "1970s",
            "reviewer_notes": null
    },
    {
            "review_no": 160,
            "review_author": "AGladyshev",
            "singular_tag": "80",
            "plural_tag": "1980s",
            "decision": "Merge",
            "merge_into": "1980s",
            "reviewer_notes": null
    },
    {
            "review_no": 161,
            "review_author": "AGladyshev",
            "singular_tag": "80's",
            "plural_tag": "1980s",
            "decision": "Merge",
            "merge_into": "1980s",
            "reviewer_notes": null
    },
    {
            "review_no": 162,
            "review_author": "AGladyshev",
            "singular_tag": "70s",
            "plural_tag": "1970s",
            "decision": "Merge",
            "merge_into": "1970s",
            "reviewer_notes": null
    },
    {
            "review_no": 163,
            "review_author": "AGladyshev",
            "singular_tag": "00s",
            "plural_tag": "2000s",
            "decision": "Merge",
            "merge_into": "2000s",
            "reviewer_notes": null
    },
    {
            "review_no": 164,
            "review_author": "AGladyshev",
            "singular_tag": "alternativ",
            "plural_tag": "alternative",
            "decision": "Merge",
            "merge_into": "alternative",
            "reviewer_notes": null
    },
    {
            "review_no": 165,
            "review_author": "AGladyshev",
            "singular_tag": "alternatif",
            "plural_tag": "alternative",
            "decision": "Merge",
            "merge_into": "alternative",
            "reviewer_notes": null
    },
    {
            "review_no": 166,
            "review_author": "AGladyshev",
            "singular_tag": "alternatif muzik",
            "plural_tag": "alternative",
            "decision": "Merge",
            "merge_into": "alternative",
            "reviewer_notes": null
    },
    {
            "review_no": 167,
            "review_author": "AGladyshev",
            "singular_tag": "rock alternatif",
            "plural_tag": "rock alternative",
            "decision": "Merge",
            "merge_into": "rock alternative",
            "reviewer_notes": null
    },
    {
            "review_no": 168,
            "review_author": "AGladyshev",
            "singular_tag": "rock alternativo",
            "plural_tag": "rock alternative",
            "decision": "Merge",
            "merge_into": "rock alternative",
            "reviewer_notes": null
    },
    {
      "review_no": 169,
      "review_author": "AGladyshev",
      "singular_tag": "synth pop",
      "plural_tag": "synthpop",
      "decision": "Merge",
      "merge_into": "synth-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 1691,
      "review_author": "AGladyshev",
      "singular_tag": "synthpop songs",
      "plural_tag": "synth-pop",
      "decision": "Merge",
      "merge_into": "synth-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 1692,
      "review_author": "AGladyshev",
      "singular_tag": "synthpop artists",
      "plural_tag": "synth-pop",
      "decision": "Merge",
      "merge_into": "synth-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 170,
      "review_author": "AGladyshev",
      "singular_tag": "hiphop",
      "plural_tag": "hip-hop",
      "decision": "Merge",
      "merge_into": "hip-hop",
      "reviewer_notes": null
    },
    {
      "review_no": 171,
      "review_author": "AGladyshev",
      "singular_tag": "trip-hop",
      "plural_tag": "trip hop",
      "decision": "Merge",
      "merge_into": "trip hop",
      "reviewer_notes": null
    },
    {
      "review_no": 172,
      "review_author": "AGladyshev",
      "singular_tag": "dream pop",
      "plural_tag": "dream-pop",
      "decision": "Merge",
      "merge_into": "dream pop",
      "reviewer_notes": null
    },
    {
      "review_no": 173,
      "review_author": "AGladyshev",
      "singular_tag": "poprock",
      "plural_tag": "pop rock",
      "decision": "Merge",
      "merge_into": "pop rock",
      "reviewer_notes": null
    },
    {
      "review_no": 174,
      "review_author": "AGladyshev",
      "singular_tag": "rock-pop",
      "plural_tag": "pop rock",
      "decision": "Merge",
      "merge_into": "pop rock",
      "reviewer_notes": null
    },
    {
      "review_no": 175,
      "review_author": "AGladyshev",
      "singular_tag": "hardrock",
      "plural_tag": "hard rock",
      "decision": "Merge",
      "merge_into": "hard rock",
      "reviewer_notes": null
    },
    {
      "review_no": 176,
      "review_author": "AGladyshev",
      "singular_tag": "punk rock",
      "plural_tag": "punkrock",
      "decision": "Merge",
      "merge_into": "punk rock",
      "reviewer_notes": null
    },
    {
      "review_no": 177,
      "review_author": "AGladyshev",
      "singular_tag": "post-rock",
      "plural_tag": "postrock",
      "decision": "Merge",
      "merge_into": "post-rock",
      "reviewer_notes": null
    },
    {
      "review_no": 178,
      "review_author": "AGladyshev",
      "singular_tag": "shoegazing",
      "plural_tag": "shoegaze",
      "decision": "Merge",
      "merge_into": "shoegaze",
      "reviewer_notes": null
    },
    {
      "review_no": 179,
      "review_author": "AGladyshev",
      "singular_tag": "japanese rock",
      "plural_tag": "j-rock",
      "decision": "Merge",
      "merge_into": "j-rock",
      "reviewer_notes": null
    },
    {
      "review_no": 180,
      "review_author": "AGladyshev",
      "singular_tag": "j rock",
      "plural_tag": "j-rock",
      "decision": "Merge",
      "merge_into": "j-rock",
      "reviewer_notes": null
    },
    {
      "review_no": 181,
      "review_author": "AGladyshev",
      "singular_tag": "japanese pop",
      "plural_tag": "j-pop",
      "decision": "Merge",
      "merge_into": "j-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 182,
      "review_author": "AGladyshev",
      "singular_tag": "jpop",
      "plural_tag": "j-pop",
      "decision": "Merge",
      "merge_into": "j-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 183,
      "review_author": "AGladyshev",
      "singular_tag": "kpop",
      "plural_tag": "k-pop",
      "decision": "Merge",
      "merge_into": "k-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 184,
      "review_author": "AGladyshev",
      "singular_tag": "rnb",
      "plural_tag": "r'n'b",
      "decision": "Merge",
      "merge_into": "r'n'b",
      "reviewer_notes": null
    },
    {
      "review_no": 185,
      "review_author": "AGladyshev",
      "singular_tag": "r and b",
      "plural_tag": "r'n'b",
      "decision": "Merge",
      "merge_into": "r'n'b",
      "reviewer_notes": null
    },
    {
      "review_no": 186,
      "review_author": "AGladyshev",
      "singular_tag": "rb",
      "plural_tag": "r'n'b",
      "decision": "Merge",
      "merge_into": "r'n'b",
      "reviewer_notes": null
    },
    {
      "review_no": 187,
      "review_author": "AGladyshev",
      "singular_tag": "drum and bass",
      "plural_tag": "d'n'b",
      "decision": "Merge",
      "merge_into": "d'n'b",
      "reviewer_notes": null
    },
    {
      "review_no": 188,
      "review_author": "AGladyshev",
      "singular_tag": "drum n bass",
      "plural_tag": "d'n'b",
      "decision": "Merge",
      "merge_into": "d'n'b",
      "reviewer_notes": null
    },
    {
      "review_no": 189,
      "review_author": "AGladyshev",
      "singular_tag": "dnb",
      "plural_tag": "d'n'b",
      "decision": "Merge",
      "merge_into": "d'n'b",
      "reviewer_notes": null
    },
    {
      "review_no": 190,
      "review_author": "AGladyshev",
      "singular_tag": "drum bass",
      "plural_tag": "d'n'b",
      "decision": "Merge",
      "merge_into": "d'n'b",
      "reviewer_notes": null
    },
    {
      "review_no": 191,
      "review_author": "AGladyshev",
      "singular_tag": "down tempo",
      "plural_tag": "downtempo",
      "decision": "Merge",
      "merge_into": "downtempo",
      "reviewer_notes": null
    },
    {
      "review_no": 192,
      "review_author": "AGladyshev",
      "singular_tag": "low tempo",
      "plural_tag": "downtempo",
      "decision": "Merge",
      "merge_into": "downtempo",
      "reviewer_notes": null
    },
    {
      "review_no": 193,
      "review_author": "AGladyshev",
      "singular_tag": "chillout",
      "plural_tag": "chill out",
      "decision": "Merge",
      "merge_into": "chill out",
      "reviewer_notes": null
    },
    {
      "review_no": 194,
      "review_author": "AGladyshev",
      "singular_tag": "chill wave",
      "plural_tag": "chillwave",
      "decision": "Merge",
      "merge_into": "chillwave",
      "reviewer_notes": null
    },
    {
      "review_no": 195,
      "review_author": "AGladyshev",
      "singular_tag": "darkwave",
      "plural_tag": "dark wave",
      "decision": "Merge",
      "merge_into": "dark wave",
      "reviewer_notes": null
    },
    {
      "review_no": 196,
      "review_author": "AGladyshev",
      "singular_tag": "ambiant",
      "plural_tag": "ambient",
      "decision": "Merge",
      "merge_into": "ambient",
      "reviewer_notes": null
    },
    {
      "review_no": 197,
      "review_author": "AGladyshev",
      "singular_tag": "ambiental",
      "plural_tag": "ambient",
      "decision": "Merge",
      "merge_into": "ambient",
      "reviewer_notes": null
    },
    {
      "review_no": 198,
      "review_author": "AGladyshev",
      "singular_tag": "electronica",
      "plural_tag": "electronic",
      "decision": "Merge",
      "merge_into": "electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 199,
      "review_author": "AGladyshev",
      "singular_tag": "eletronica",
      "plural_tag": "electronic",
      "decision": "Merge",
      "merge_into": "electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 200,
      "review_author": "AGladyshev",
      "singular_tag": "elertronic",
      "plural_tag": "electronic",
      "decision": "Merge",
      "merge_into": "electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 201,
      "review_author": "AGladyshev",
      "singular_tag": "eletronic",
      "plural_tag": "electronic",
      "decision": "Merge",
      "merge_into": "electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 202,
      "review_author": "AGladyshev",
      "singular_tag": "elecronica",
      "plural_tag": "electronic",
      "decision": "Merge",
      "merge_into": "electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 203,
      "review_author": "AGladyshev",
      "singular_tag": "electro pop",
      "plural_tag": "electronic-pop",
      "decision": "Merge",
      "merge_into": "electronic-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 204,
      "review_author": "AGladyshev",
      "singular_tag": "eletropop",
      "plural_tag": "electronic-pop",
      "decision": "Merge",
      "merge_into": "electronic-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 205,
      "review_author": "AGladyshev",
      "singular_tag": "eletro pop",
      "plural_tag": "electronic-pop",
      "decision": "Merge",
      "merge_into": "electronic-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 206,
      "review_author": "AGladyshev",
      "singular_tag": "acustic",
      "plural_tag": "acoustic",
      "decision": "Merge",
      "merge_into": "acoustic",
      "reviewer_notes": null
    },
    {
      "review_no": 207,
      "review_author": "AGladyshev",
      "singular_tag": "acustico",
      "plural_tag": "acoustic",
      "decision": "Merge",
      "merge_into": "acoustic",
      "reviewer_notes": null
    },
    {
      "review_no": 208,
      "review_author": "AGladyshev",
      "singular_tag": "visualkei",
      "plural_tag": "visual kei",
      "decision": "Merge",
      "merge_into": "visual kei",
      "reviewer_notes": null
    },
    {
      "review_no": 209,
      "review_author": "AGladyshev",
      "singular_tag": "rock and roll",
      "plural_tag": "rock'n'roll",
      "decision": "Merge",
      "merge_into": "rock'n'roll",
      "reviewer_notes": null
    },
    {
      "review_no": 210,
      "review_author": "AGladyshev",
      "singular_tag": "rock n roll",
      "plural_tag": "rock'n'roll",
      "decision": "Merge",
      "merge_into": "rock'n'roll",
      "reviewer_notes": null
    },
    {
      "review_no": 211,
      "review_author": "AGladyshev",
      "singular_tag": "rock 'n' roll",
      "plural_tag": "rock'n'roll",
      "decision": "Merge",
      "merge_into": "rock'n'roll",
      "reviewer_notes": null
    },
    {
      "review_no": 212,
      "review_author": "AGladyshev",
      "singular_tag": "rocknroll",
      "plural_tag": "rock'n'roll",
      "decision": "Merge",
      "merge_into": "rock'n'roll",
      "reviewer_notes": null
    },
    {
      "review_no": 213,
      "review_author": "AGladyshev",
      "singular_tag": "acapella",
      "plural_tag": "a cappella",
      "decision": "Merge",
      "merge_into": "a cappella",
      "reviewer_notes": null
    },
    {
      "review_no": 214,
      "review_author": "AGladyshev",
      "singular_tag": "a capella",
      "plural_tag": "a cappella",
      "decision": "Merge",
      "merge_into": "a cappella",
      "reviewer_notes": null
    },
    {
      "review_no": 215,
      "review_author": "AGladyshev",
      "singular_tag": "cristian rock",
      "plural_tag": "christian rock",
      "decision": "Merge",
      "merge_into": "christian rock",
      "reviewer_notes": null
    },
    {
      "review_no": 216,
      "review_author": "AGladyshev",
      "singular_tag": "steve abini",
      "plural_tag": "steve albini",
      "decision": "Merge",
      "merge_into": "steve albini",
      "reviewer_notes": null
    },
    {
      "review_no": 217,
      "review_author": "AGladyshev",
      "singular_tag": "psychadelic",
      "plural_tag": "psychedelic",
      "decision": "Merge",
      "merge_into": "psychedelic",
      "reviewer_notes": null
    },
    {
      "review_no": 218,
      "review_author": "AGladyshev",
      "singular_tag": "tecno",
      "plural_tag": "techno",
      "decision": "Merge",
      "merge_into": "techno",
      "reviewer_notes": null
    },
    {
      "review_no": 219,
      "review_author": "AGladyshev",
      "singular_tag": "relaxing",
      "plural_tag": "relax",
      "decision": "Merge",
      "merge_into": "relax",
      "reviewer_notes": null
    },
    {
      "review_no": 220,
      "review_author": "AGladyshev",
      "singular_tag": "relaxed",
      "plural_tag": "relax",
      "decision": "Merge",
      "merge_into": "relax",
      "reviewer_notes": null
    },
    {
      "review_no": 221,
      "review_author": "AGladyshev",
      "singular_tag": "relaxxx",
      "plural_tag": "relax",
      "decision": "Merge",
      "merge_into": "relax",
      "reviewer_notes": null
    },
    {
      "review_no": 222,
      "review_author": "AGladyshev",
      "singular_tag": "80's",
      "plural_tag": "1980s",
      "decision": "Merge",
      "merge_into": "1980s",
      "reviewer_notes": null
    },
    {
      "review_no": 223,
      "review_author": "AGladyshev",
      "singular_tag": "1979 songs",
      "plural_tag": "1979",
      "decision": "Merge",
      "merge_into": "1979",
      "reviewer_notes": null
    },
    {
      "review_no": 224,
      "review_author": "AGladyshev",
      "singular_tag": "new wave artists",
      "plural_tag": "new wave",
      "decision": "Merge",
      "merge_into": "new wave",
      "reviewer_notes": null
    },
    {
      "review_no": 225,
      "review_author": "AGladyshev",
      "singular_tag": "new wave songs",
      "plural_tag": "new wave",
      "decision": "Merge",
      "merge_into": "new wave",
      "reviewer_notes": null
    },
    {
      "review_no": 226,
      "review_author": "AGladyshev",
      "singular_tag": "rock brasileiro",
      "plural_tag": "rock brasil",
      "decision": "Merge",
      "merge_into": "rock brasil",
      "reviewer_notes": null
    },
    {
      "review_no": 227,
      "review_author": "AGladyshev",
      "singular_tag": "brasileira",
      "plural_tag": "brazilian",
      "decision": "Merge",
      "merge_into": "brazilian",
      "reviewer_notes": null
    },
    {
      "review_no": 228,
      "review_author": "AGladyshev",
      "singular_tag": "60's",
      "plural_tag": "1960s",
      "decision": "Merge",
      "merge_into": "1960s",
      "reviewer_notes": null
    },
    {
      "review_no": 229,
      "review_author": "AGladyshev",
      "singular_tag": "80's music",
      "plural_tag": "1980s",
      "decision": "Merge",
      "merge_into": "1980s",
      "reviewer_notes": null
    },
    {
      "review_no": 230,
      "review_author": "AGladyshev",
      "singular_tag": "90's",
      "plural_tag": "1990s",
      "decision": "Merge",
      "merge_into": "1990s",
      "reviewer_notes": null
    },
    {
      "review_no": 231,
      "review_author": "AGladyshev",
      "singular_tag": "80s",
      "plural_tag": "1980s",
      "decision": "Merge",
      "merge_into": "1980s",
      "reviewer_notes": null
    },
    {
      "review_no": 232,
      "review_author": "AGladyshev",
      "singular_tag": "50's",
      "plural_tag": "1950s",
      "decision": "Merge",
      "merge_into": "1950s",
      "reviewer_notes": null
    },
    {
      "review_no": 233,
      "review_author": "AGladyshev",
      "singular_tag": "50s",
      "plural_tag": "1950s",
      "decision": "Merge",
      "merge_into": "1950s",
      "reviewer_notes": null
    },
    {
      "review_no": 234,
      "review_author": "AGladyshev",
      "singular_tag": "60s",
      "plural_tag": "1960s",
      "decision": "Merge",
      "merge_into": "1960s",
      "reviewer_notes": null
    },
    {
      "review_no": 235,
      "review_author": "AGladyshev",
      "singular_tag": "acdc",
      "plural_tag": "ac dc",
      "decision": "Merge",
      "merge_into": "ac dc",
      "reviewer_notes": null
    },
    {
      "review_no": 236,
      "review_author": "AGladyshev",
      "singular_tag": "alternativee",
      "plural_tag": "alternative",
      "decision": "Merge",
      "merge_into": "alternative",
      "reviewer_notes": null
    },
    {
      "review_no": 237,
      "review_author": "AGladyshev",
      "singular_tag": "alternativo",
      "plural_tag": "alternative",
      "decision": "Merge",
      "merge_into": "alternative",
      "reviewer_notes": null
    },
    {
      "review_no": 238,
      "review_author": "AGladyshev",
      "singular_tag": "alt",
      "plural_tag": "alternative",
      "decision": "Merge",
      "merge_into": "alternative",
      "reviewer_notes": null
    },
    {
      "review_no": 239,
      "review_author": "AGladyshev",
      "singular_tag": "alt rock",
      "plural_tag": "alternative rock",
      "decision": "Merge",
      "merge_into": "alternative rock",
      "reviewer_notes": null
    },
    {
      "review_no": 240,
      "review_author": "AGladyshev",
      "singular_tag": "alt-country",
      "plural_tag": "alternative country",
      "decision": "Merge",
      "merge_into": "alternative country",
      "reviewer_notes": null
    },
    {
      "review_no": 241,
      "review_author": "AGladyshev",
      "singular_tag": "alger",
      "plural_tag": "algeria",
      "decision": "Merge",
      "merge_into": "algeria",
      "reviewer_notes": null
    },
    {
      "review_no": 242,
      "review_author": "AGladyshev",
      "singular_tag": "anos 80",
      "plural_tag": "1980s",
      "decision": "Merge",
      "merge_into": "1980s",
      "reviewer_notes": null
    },
    {
      "review_no": 243,
      "review_author": "AGladyshev",
      "singular_tag": "anthemic",
      "plural_tag": "anthem",
      "decision": "Merge",
      "merge_into": "anthem",
      "reviewer_notes": null
    },
    {
      "review_no": 244,
      "review_author": "AGladyshev",
      "singular_tag": "arabica",
      "plural_tag": "arabic",
      "decision": "Merge",
      "merge_into": "arabic",
      "reviewer_notes": null
    },
    {
      "review_no": 245,
      "review_author": "AGladyshev",
      "singular_tag": "australia",
      "plural_tag": "australian",
      "decision": "Merge",
      "merge_into": "australian",
      "reviewer_notes": null
    },
    {
      "review_no": 246,
      "review_author": "AGladyshev",
      "singular_tag": "avantgarde",
      "plural_tag": "avant-garde",
      "decision": "Merge",
      "merge_into": "avant-garde",
      "reviewer_notes": null
    },
    {
      "review_no": 247,
      "review_author": "AGladyshev",
      "singular_tag": "beach",
      "plural_tag": "beach music",
      "decision": "Merge",
      "merge_into": "beach music",
      "reviewer_notes": null
    },
    {
      "review_no": 248,
      "review_author": "AGladyshev",
      "singular_tag": "belarus",
      "plural_tag": "belarusian",
      "decision": "Merge",
      "merge_into": "belarusian",
      "reviewer_notes": null
    },
    {
      "review_no": 249,
      "review_author": "AGladyshev",
      "singular_tag": "bitter sweet",
      "plural_tag": "bittersweet",
      "decision": "Merge",
      "merge_into": "bittersweet",
      "reviewer_notes": null
    },
    {
      "review_no": 250,
      "review_author": "AGladyshev",
      "singular_tag": "brasil",
      "plural_tag": "brazil",
      "decision": "Merge",
      "merge_into": "brazil",
      "reviewer_notes": null
    },
    {
      "review_no": 251,
      "review_author": "AGladyshev",
      "singular_tag": "brasilian",
      "plural_tag": "brazilian",
      "decision": "Merge",
      "merge_into": "brazilian",
      "reviewer_notes": null
    },
    {
      "review_no": 252,
      "review_author": "AGladyshev",
      "singular_tag": "brit rock",
      "plural_tag": "british rock",
      "decision": "Merge",
      "merge_into": "british rock",
      "reviewer_notes": null
    },
    {
      "review_no": 253,
      "review_author": "AGladyshev",
      "singular_tag": "britney",
      "plural_tag": "britney spears",
      "decision": "Merge",
      "merge_into": "britney spears",
      "reviewer_notes": null
    },
    {
      "review_no": 254,
      "review_author": "AGladyshev",
      "singular_tag": "britneyspears",
      "plural_tag": "britney spears",
      "decision": "Merge",
      "merge_into": "britney spears",
      "reviewer_notes": null
    },
    {
      "review_no": 255,
      "review_author": "AGladyshev",
      "singular_tag": "britrock",
      "plural_tag": "british rock",
      "decision": "Merge",
      "merge_into": "british rock",
      "reviewer_notes": null
    },
    {
      "review_no": 256,
      "review_author": "AGladyshev",
      "singular_tag": "britpop",
      "plural_tag": "brit pop",
      "decision": "Merge",
      "merge_into": "brit pop",
      "reviewer_notes": null
    },
    {
      "review_no": 257,
      "review_author": "AGladyshev",
      "singular_tag": "central asia",
      "plural_tag": "central asian",
      "decision": "Merge",
      "merge_into": "central asian",
      "reviewer_notes": null
    },
    {
      "review_no": 258,
      "review_author": "AGladyshev",
      "singular_tag": "classico",
      "plural_tag": "classic",
      "decision": "Merge",
      "merge_into": "classic",
      "reviewer_notes": null
    },
    {
      "review_no": 259,
      "review_author": "AGladyshev",
      "singular_tag": "clasica",
      "plural_tag": "classic",
      "decision": "Merge",
      "merge_into": "classic",
      "reviewer_notes": null
    },
    {
      "review_no": 260,
      "review_author": "AGladyshev",
      "singular_tag": "country: britain",
      "plural_tag": "british",
      "decision": "Merge",
      "merge_into": "british",
      "reviewer_notes": null
    },
    {
      "review_no": 261,
      "review_author": "AGladyshev",
      "singular_tag": "male vocals",
      "plural_tag": "male vocalist",
      "decision": "Merge",
      "merge_into": "male vocalist",
      "reviewer_notes": null
    },
    {
      "review_no": 262,
      "review_author": "AGladyshev",
      "singular_tag": "female vocals",
      "plural_tag": "female vocalist",
      "decision": "Merge",
      "merge_into": "female vocalist",
      "reviewer_notes": null
    },
    {
      "review_no": 263,
      "review_author": "AGladyshev",
      "singular_tag": "female voice",
      "plural_tag": "female vocalist",
      "decision": "Merge",
      "merge_into": "female vocalist",
      "reviewer_notes": null
    },
    {
      "review_no": 264,
      "review_author": "AGladyshev",
      "singular_tag": "funky",
      "plural_tag": "funk",
      "decision": "Merge",
      "merge_into": "funk",
      "reviewer_notes": null
    },
    {
      "review_no": 265,
      "review_author": "AGladyshev",
      "singular_tag": "funkalistic",
      "plural_tag": "funkalistik",
      "decision": "Merge",
      "merge_into": "funkalistik",
      "reviewer_notes": null
    },
    {
      "review_no": 266,
      "review_author": "AGladyshev",
      "singular_tag": "horrorpunk",
      "plural_tag": "horror punk",
      "decision": "Merge",
      "merge_into": "horror punk",
      "reviewer_notes": null
    },
    {
      "review_no": 267,
      "review_author": "AGladyshev",
      "singular_tag": "indierock",
      "plural_tag": "indie rock",
      "decision": "Merge",
      "merge_into": "indie rock",
      "reviewer_notes": null
    },
    {
      "review_no": 268,
      "review_author": "AGladyshev",
      "singular_tag": "indie electronica",
      "plural_tag": "indie electronic",
      "decision": "Merge",
      "merge_into": "indie electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 269,
      "review_author": "AGladyshev",
      "singular_tag": "indietronica",
      "plural_tag": "indie electronic",
      "decision": "Merge",
      "merge_into": "indie electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 270,
      "review_author": "AGladyshev",
      "singular_tag": "indietronic",
      "plural_tag": "indie electronic",
      "decision": "Merge",
      "merge_into": "indie electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 271,
      "review_author": "AGladyshev",
      "singular_tag": "industri",
      "plural_tag": "industrial",
      "decision": "Merge",
      "merge_into": "industrial",
      "reviewer_notes": null
    },
    {
      "review_no": 272,
      "review_author": "AGladyshev",
      "singular_tag": "instro rock",
      "plural_tag": "instrumental rock",
      "decision": "Merge",
      "merge_into": "instrumental rock",
      "reviewer_notes": null
    },
    {
      "review_no": 273,
      "review_author": "AGladyshev",
      "singular_tag": "italiano",
      "plural_tag": "italian",
      "decision": "Merge",
      "merge_into": "italian",
      "reviewer_notes": null
    },
    {
      "review_no": 274,
      "review_author": "AGladyshev",
      "singular_tag": "italiana",
      "plural_tag": "italian",
      "decision": "Merge",
      "merge_into": "italian",
      "reviewer_notes": null
    },
    {
      "review_no": 275,
      "review_author": "AGladyshev",
      "singular_tag": "kazakistan",
      "plural_tag": "kazakhstan",
      "decision": "Merge",
      "merge_into": "kazakhstan",
      "reviewer_notes": null
    },
    {
      "review_no": 276,
      "review_author": "AGladyshev",
      "singular_tag": "vocalista feminimo",
      "plural_tag": "female vocalist",
      "decision": "Merge",
      "merge_into": "female vocalist",
      "reviewer_notes": null
    },
    {
      "review_no": 277,
      "review_author": "AGladyshev",
      "singular_tag": "vocal femenino",
      "plural_tag": "female vocalist",
      "decision": "Merge",
      "merge_into": "female vocalist",
      "reviewer_notes": null
    },
    {
      "review_no": 278,
      "review_author": "AGladyshev",
      "singular_tag": "xmas",
      "plural_tag": "x-mas",
      "decision": "Merge",
      "merge_into": "x-mas",
      "reviewer_notes": null
    },
    {
      "review_no": 279,
      "review_author": "AGladyshev",
      "singular_tag": "christmas blend",
      "plural_tag": "christmas",
      "decision": "Merge",
      "merge_into": "christmas",
      "reviewer_notes": null
    },
    {
      "review_no": 280,
      "review_author": "AGladyshev",
      "singular_tag": "christmas song",
      "plural_tag": "christmas",
      "decision": "Merge",
      "merge_into": "christmas",
      "reviewer_notes": null
    },
    {
      "review_no": 281,
      "review_author": "AGladyshev",
      "singular_tag": "europop",
      "plural_tag": "euro-pop",
      "decision": "Merge",
      "merge_into": "euro-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 282,
      "review_author": "AGladyshev",
      "singular_tag": "eurovision song contest",
      "plural_tag": "eurovision",
      "decision": "Merge",
      "merge_into": "eurovision",
      "reviewer_notes": null
    },
    {
      "review_no": 283,
      "review_author": "AGladyshev",
      "singular_tag": "guns n' roses",
      "plural_tag": "guns and roses",
      "decision": "Merge",
      "merge_into": "guns and roses",
      "reviewer_notes": null
    },
    {
      "review_no": 284,
      "review_author": "AGladyshev",
      "singular_tag": "powerfull",
      "plural_tag": "powerful",
      "decision": "Merge",
      "merge_into": "powerful",
      "reviewer_notes": null
    },
    {
      "review_no": 285,
      "review_author": "AGladyshev",
      "singular_tag": "powerfull voices",
      "plural_tag": "powerful voice",
      "decision": "Merge",
      "merge_into": "powerful voice",
      "reviewer_notes": null
    },
    {
      "review_no": 286,
      "review_author": "AGladyshev",
      "singular_tag": "power voice",
      "plural_tag": "powerful voice",
      "decision": "Merge",
      "merge_into": "powerful voice",
      "reviewer_notes": null
    },
    {
      "review_no": 287,
      "review_author": "AGladyshev",
      "singular_tag": "prog",
      "plural_tag": "progressive",
      "decision": "Merge",
      "merge_into": "progressive",
      "reviewer_notes": null
    },
    {
      "review_no": 288,
      "review_author": "AGladyshev",
      "singular_tag": "prog folk",
      "plural_tag": "progressive folk",
      "decision": "Merge",
      "merge_into": "progressive folk",
      "reviewer_notes": null
    },
    {
      "review_no": 289,
      "review_author": "AGladyshev",
      "singular_tag": "prog metal",
      "plural_tag": "progressive metal",
      "decision": "Merge",
      "merge_into": "progressive metal",
      "reviewer_notes": null
    },
    {
      "review_no": 290,
      "review_author": "AGladyshev",
      "singular_tag": "prog rock",
      "plural_tag": "progressive rock",
      "decision": "Merge",
      "merge_into": "progressive rock",
      "reviewer_notes": null
    },
    {
      "review_no": 291,
      "review_author": "AGladyshev",
      "singular_tag": "rolling stone 500 greatest songs of all time",
      "plural_tag": "rolling stones top 500",
      "decision": "Merge",
      "merge_into": "rolling stones top 500",
      "reviewer_notes": null
    },
    {
      "review_no": 292,
      "review_author": "AGladyshev",
      "singular_tag": "synthrock",
      "plural_tag": "synth-rock",
      "decision": "Merge",
      "merge_into": "synth-rock",
      "reviewer_notes": null
    },
    {
      "review_no": 293,
      "review_author": "AGladyshev",
      "singular_tag": "talent",
      "plural_tag": "talented",
      "decision": "Merge",
      "merge_into": "talented",
      "reviewer_notes": null
    },
    {
      "review_no": 294,
      "review_author": "AGladyshev",
      "singular_tag": "teenpop",
      "plural_tag": "teen pop",
      "decision": "Merge",
      "merge_into": "teen pop",
      "reviewer_notes": null
    },
    {
      "review_no": 295,
      "review_author": "AGladyshev",
      "singular_tag": "werewolves",
      "plural_tag": "werewolf",
      "decision": "Merge",
      "merge_into": "werewolf",
      "reviewer_notes": null
    },
    {
      "review_no": 296,
      "review_author": "AGladyshev",
      "singular_tag": "top40",
      "plural_tag": "top 40",
      "decision": "Merge",
      "merge_into": "top 40",
      "reviewer_notes": null
    },
    {
      "review_no": 297,
      "review_author": "AGladyshev",
      "singular_tag": "synthesizer",
      "plural_tag": "synth",
      "decision": "Merge",
      "merge_into": "synth",
      "reviewer_notes": null
    },
    {
      "review_no": 298,
      "review_author": "AGladyshev",
      "singular_tag": "elephant six",
      "plural_tag": "elephant 6",
      "decision": "Merge",
      "merge_into": "elephant 6",
      "reviewer_notes": null
    },
    {
      "review_no": 299,
      "review_author": "AGladyshev",
      "singular_tag": "eletro",
      "plural_tag": "electronic",
      "decision": "Merge",
      "merge_into": "electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 300,
      "review_author": "AGladyshev",
      "singular_tag": "electro",
      "plural_tag": "electronic",
      "decision": "Merge",
      "merge_into": "electronic",
      "reviewer_notes": null
    },
    {
      "review_no": 301,
      "review_author": "AGladyshev",
      "singular_tag": "electropop",
      "plural_tag": "electronic-pop",
      "decision": "Merge",
      "merge_into": "electronic-pop",
      "reviewer_notes": null
    },
    {
      "review_no": 302,
      "review_author": "AGladyshev",
      "singular_tag": "electropunk",
      "plural_tag": "electro punk",
      "decision": "Merge",
      "merge_into": "electro punk",
      "reviewer_notes": null
    },
    {
      "review_no": 303,
      "review_author": "AGladyshev",
      "singular_tag": "electrorock",
      "plural_tag": "electronic rock",
      "decision": "Merge",
      "merge_into": "electronic rock",
      "reviewer_notes": null
    },
    {
      "review_no": 304,
      "review_author": "AGladyshev",
      "singular_tag": "electro-rock",
      "plural_tag": "electronic rock",
      "decision": "Merge",
      "merge_into": "electronic rock",
      "reviewer_notes": null
    },
    {
      "review_no": 305,
      "review_author": "AGladyshev",
      "singular_tag": "folkloric",
      "plural_tag": "folk",
      "decision": "Merge",
      "merge_into": "folk",
      "reviewer_notes": null
    },
    {
      "review_no": 306,
      "review_author": "AGladyshev",
      "singular_tag": "folktronica",
      "plural_tag": "folktronic",
      "decision": "Merge",
      "merge_into": "folktronic",
      "reviewer_notes": null
    },
    {
      "review_no": 307,
      "review_author": "AGladyshev",
      "singular_tag": "forever in heart",
      "plural_tag": "forever",
      "decision": "Merge",
      "merge_into": "forever",
      "reviewer_notes": null
    },
    {
      "review_no": 308,
      "review_author": "AGladyshev",
      "singular_tag": "fucking awensome",
      "plural_tag": "fucking awesome",
      "decision": "Merge",
      "merge_into": "fucking awesome",
      "reviewer_notes": null
    },
    {
      "review_no": 309,
      "review_author": "AGladyshev",
      "singular_tag": "goth",
      "plural_tag": "gothic",
      "decision": "Merge",
      "merge_into": "gothic",
      "reviewer_notes": null
    },
    {
      "review_no": 310,
      "review_author": "AGladyshev",
      "singular_tag": "goth rock",
      "plural_tag": "gothic rock",
      "decision": "Merge",
      "merge_into": "gothic rock",
      "reviewer_notes": null
    },
    {
      "review_no": 311,
      "review_author": "AGladyshev",
      "singular_tag": "greys anatomy",
      "plural_tag": "greys anatomy soundtrack",
      "decision": "Merge",
      "merge_into": "greys anatomy soundtrack",
      "reviewer_notes": null
    },
    {
      "review_no": 312,
      "review_author": "AGladyshev",
      "singular_tag": "gta iv",
      "plural_tag": "gta4",
      "decision": "Merge",
      "merge_into": "gta4",
      "reviewer_notes": null
    },
    {
      "review_no": 313,
      "review_author": "AGladyshev",
      "singular_tag": "irish music",
      "plural_tag": "irish",
      "decision": "Merge",
      "merge_into": "irish",
      "reviewer_notes": null
    },
    {
      "review_no": 314,
      "review_author": "AGladyshev",
      "singular_tag": "italia",
      "plural_tag": "italy",
      "decision": "Merge",
      "merge_into": "italy",
      "reviewer_notes": null
    },
    {
      "review_no": 315,
      "review_author": "AGladyshev",
      "singular_tag": "kickass",
      "plural_tag": "kick ass",
      "decision": "Merge",
      "merge_into": "kick ass",
      "reviewer_notes": null
    },
    {
      "review_no": 316,
      "review_author": "AGladyshev",
      "singular_tag": "lovesongs",
      "plural_tag": "love song",
      "decision": "Merge",
      "merge_into": "love song",
      "reviewer_notes": null
    },
    {
      "review_no": 317,
      "review_author": "AGladyshev",
      "singular_tag": "mexicano",
      "plural_tag": "mexican",
      "decision": "Merge",
      "merge_into": "mexican",
      "reviewer_notes": null
    },
    {
      "review_no": 318,
      "review_author": "AGladyshev",
      "singular_tag": "multiculti",
      "plural_tag": "multicultural",
      "decision": "Merge",
      "merge_into": "multicultural",
      "reviewer_notes": null
    },
    {
      "review_no": 319,
      "review_author": "AGladyshev",
      "singular_tag": "multikulti",
      "plural_tag": "multicultural",
      "decision": "Merge",
      "merge_into": "multicultural",
      "reviewer_notes": null
    },
    {
      "review_no": 320,
      "review_author": "AGladyshev",
      "singular_tag": "orient",
      "plural_tag": "oriental",
      "decision": "Merge",
      "merge_into": "oriental",
      "reviewer_notes": null
    },
    {
      "review_no": 321,
      "review_author": "AGladyshev",
      "singular_tag": "partytime",
      "plural_tag": "party time",
      "decision": "Merge",
      "merge_into": "party time",
      "reviewer_notes": null
    },
    {
      "review_no": 322,
      "review_author": "AGladyshev",
      "singular_tag": "pop bitches",
      "plural_tag": "pop bitch",
      "decision": "Merge",
      "merge_into": "pop bitch",
      "reviewer_notes": null
    },
    {
      "review_no": 323,
      "review_author": "AGladyshev",
      "singular_tag": "riot grrl",
      "plural_tag": "riot grrrl",
      "decision": "Merge",
      "merge_into": "riot grrrl",
      "reviewer_notes": null
    },
    {
      "review_no": 324,
      "review_author": "AGladyshev",
      "singular_tag": "rick villa",
      "plural_tag": "rick villy villa",
      "decision": "Merge",
      "merge_into": "rick villy villa",
      "reviewer_notes": null
    },
    {
      "review_no": 325,
      "review_author": "AGladyshev",
      "singular_tag": "remixed",
      "plural_tag": "remix",
      "decision": "Merge",
      "merge_into": "remix",
      "reviewer_notes": null
    },
    {
      "review_no": 326,
      "review_author": "AGladyshev",
      "singular_tag": "remixes",
      "plural_tag": "remix",
      "decision": "Merge",
      "merge_into": "remix",
      "reviewer_notes": null
    },
    {
      "review_no": 327,
      "review_author": "AGladyshev",
      "singular_tag": "religion",
      "plural_tag": "religious",
      "decision": "Merge",
      "merge_into": "religious",
      "reviewer_notes": null
    },
    {
      "review_no": 328,
      "review_author": "AGladyshev",
      "singular_tag": "screamo",
      "plural_tag": "scream",
      "decision": "Merge",
      "merge_into": "scream",
      "reviewer_notes": null
    },
    {
      "review_no": 329,
      "review_author": "AGladyshev",
      "singular_tag": "sertanejo",
      "plural_tag": "sertaneja",
      "decision": "Merge",
      "merge_into": "sertaneja",
      "reviewer_notes": null
    },
    {
      "review_no": 330,
      "review_author": "AGladyshev",
      "singular_tag": "spacerock",
      "plural_tag": "space rock",
      "decision": "Merge",
      "merge_into": "space rock",
      "reviewer_notes": null
    },
    {
      "review_no": 331,
      "review_author": "AGladyshev",
      "singular_tag": "viola",
      "plural_tag": "violin",
      "decision": "Merge",
      "merge_into": "violin",
      "reviewer_notes": null
    },
    {
      "review_no": 332,
      "review_author": "AGladyshev",
      "singular_tag": "violinist",
      "plural_tag": "violin",
      "decision": "Merge",
      "merge_into": "violin",
      "reviewer_notes": null
    },
    {
      "review_no": 333,
      "review_author": "AGladyshev",
      "singular_tag": "wake up song",
      "plural_tag": "wake up",
      "decision": "Merge",
      "merge_into": "wake up",
      "reviewer_notes": null
    },
    {
      "review_no": 334,
      "review_author": "AGladyshev",
      "singular_tag": "video game",
      "plural_tag": "video game music",
      "decision": "Merge",
      "merge_into": "video game music",
      "reviewer_notes": null
    },
    {
      "review_no": 335,
      "review_author": "AGladyshev",
      "singular_tag": "ukrainian music",
      "plural_tag": "ukrainian",
      "decision": "Merge",
      "merge_into": "ukrainian",
      "reviewer_notes": null
    },
    {
      "review_no": 336,
      "review_author": "AGladyshev",
      "singular_tag": "suicide songs",
      "plural_tag": "suicide",
      "decision": "Merge",
      "merge_into": "suicide",
      "reviewer_notes": null
    },
    {
      "review_no": 337,
      "review_author": "AGladyshev",
      "singular_tag": "stand-up comedy",
      "plural_tag": "stand-up",
      "decision": "Merge",
      "merge_into": "stand-up",
      "reviewer_notes": null
    },
    {
      "review_no": 338,
      "review_author": "AGladyshev",
      "singular_tag": "diego12",
      "plural_tag": "diego 12",
      "decision": "Merge",
      "merge_into": "diego 12",
      "reviewer_notes": null
    },
    {
      "review_no": 339,
      "review_author": "AGladyshev",
      "singular_tag": "bitch song",
      "plural_tag": "bitch",
      "decision": "Merge",
      "merge_into": "bitch",
      "reviewer_notes": null
    },
    {
      "review_no": 340,
      "review_author": "AGladyshev",
      "singular_tag": "bitches",
      "plural_tag": "bitch",
      "decision": "Merge",
      "merge_into": "bitch",
      "reviewer_notes": null
    },
    {
      "review_no": 341,
      "review_author": "AGladyshev",
      "singular_tag": "bitchy lyrics",
      "plural_tag": "bitch",
      "decision": "Merge",
      "merge_into": "bitch",
      "reviewer_notes": null
    }


]"""))
# Normalize missing notes to empty strings so truthiness checks behave
# identically under pandas 2 (None) and pandas 3 (NaN).
APPROVED_PLURAL_REVIEW["reviewer_notes"] = (
    APPROVED_PLURAL_REVIEW["reviewer_notes"].fillna("")
)

FLAGGED_REVIEW_NUMBERS = set([9, 13, 15, 16, 22, 24, 27, 29, 30, 32, 34, 36, 42, 43, 50, 51, 52, 53, 57, 59, 63, 64, 68, 69, 70, 71, 73, 74, 75, 76, 77, 78, 80, 83, 84, 86, 87, 88, 91, 95, 96, 97, 98, 99, 101, 102, 103, 104, 106, 107, 108, 109, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 122, 123, 124, 126, 128, 130, 133, 134, 135, 137, 138])


assert (APPROVED_PLURAL_REVIEW["decision"] == "Keep separate").sum() == 22
assert len(FLAGGED_REVIEW_NUMBERS) == 73

APPROVED_PLURAL_REVIEW["is_flagged_review_pair"] = (
    APPROVED_PLURAL_REVIEW["review_no"].isin(FLAGGED_REVIEW_NUMBERS)
)

display(
    APPROVED_PLURAL_REVIEW[
        ["decision", "is_flagged_review_pair"]
    ].value_counts().rename("candidate_pairs").to_frame()
)

candidate_pairs
decision      is_flagged_review_pair                 
Merge         False                               262
              True                                 59
Keep separate True                                 14
              False                                 8

In [797]:
available_base_keys = set(tag_mapping["tag_key"])

review_source_keys = set()
plural_key_map = {}
preferred_display_by_key = {}

for _, row in APPROVED_PLURAL_REVIEW.iterrows():
    singular_key = make_tag_key(row["singular_tag"])
    plural_key = make_tag_key(row["plural_tag"])
    review_source_keys.update([singular_key, plural_key])

    if row["decision"] == "Merge":
        target_key = make_tag_key(row["merge_into"])
        plural_key_map[singular_key] = target_key
        plural_key_map[plural_key] = target_key
        preferred_display_by_key[target_key] = clean_tag_text(row["merge_into"])

missing_review_keys = sorted(review_source_keys - available_base_keys)
assert not missing_review_keys, f"Approved review keys missing from tags.dat: {missing_review_keys}"

tag_mapping["post_plural_key"] = tag_mapping["tag_key"].map(
    lambda key: plural_key_map.get(key, key)
)

print(
    "Keys after approved singular/plural decisions:",
    f"{tag_mapping['post_plural_key'].nunique():,}",
)

Keys after approved singular/plural decisions: 11,490


In [798]:
md("""## Chapter 17 - Apply conservative spelling-equivalence mappings

Only four additional equivalences were approved:

```text
favourite -> favorite
fav -> fave
my favourite -> my favorite
boysband -> boyband
```

These correct demonstrably equivalent spellings, abbreviations, or malformed forms.

Broader semantic relatives remain separate. For example:

```text
female vocalist != female vocals
soundtrack != game soundtrack
favorite != favorite song
girl group != girl band
```""")

## Chapter 17 - Apply conservative spelling-equivalence mappings

Only four additional equivalences were approved:

```text
favourite -> favorite
fav -> fave
my favourite -> my favorite
boysband -> boyband
```

These correct demonstrably equivalent spellings, abbreviations, or malformed forms.

Broader semantic relatives remain separate. For example:

```text
female vocalist != female vocals
soundtrack != game soundtrack
favorite != favorite song
girl group != girl band
```

In [799]:
CONSERVATIVE_EQUIVALENCES = {
    "favourite": "favorite",
    "fav": "fave",
    "my favourite": "my favorite",
    "boysband": "boyband",
}

equivalence_key_map = {
    make_tag_key(source): make_tag_key(target)
    for source, target in CONSERVATIVE_EQUIVALENCES.items()
}

for target in CONSERVATIVE_EQUIVALENCES.values():
    preferred_display_by_key[make_tag_key(target)] = clean_tag_text(target)

tag_mapping["canonical_key"] = tag_mapping["post_plural_key"].map(
    lambda key: equivalence_key_map.get(key, key)
)

canonical_display_by_key = {}
for canonical_key in tag_mapping["canonical_key"].unique():
    if canonical_key in preferred_display_by_key:
        canonical_display_by_key[canonical_key] = preferred_display_by_key[canonical_key]
    elif canonical_key in base_display_by_key.index:
        canonical_display_by_key[canonical_key] = base_display_by_key.loc[canonical_key]
    else:
        canonical_display_by_key[canonical_key] = canonical_key

tag_mapping["canonical_tag"] = tag_mapping["canonical_key"].map(
    canonical_display_by_key
)

assert tag_mapping["canonical_tag"].notna().all()

print(f"Raw tag IDs:        {len(tag_mapping):,}")
print(f"Final tag keys:     {tag_mapping['canonical_key'].nunique():,}")
display(
    tag_mapping.loc[
        tag_mapping["tag_key"] != tag_mapping["canonical_key"],
        ["tagID", "raw_tag", "tag_key", "canonical_key", "canonical_tag"],
    ].head(20)
)

Raw tag IDs:        11,946
Final tag keys:     11,486


,tagID,raw_tag,tag_key,canonical_key,canonical_tag
2,3,goth rock,goth rock,gothic rock,gothic rock
12,13,chillout,chillout,chill out,chill out
18,19,80's,80's,1980s,1980s
24,25,80s,80s,1980s,1980s
64,65,folktronica,folktronica,folktronic,folktronic
70,71,hiphop,hiphop,hip hop,hip-hop
73,74,synthpop,synthpop,synth pop,synth-pop
107,108,rock and roll,rock and roll,rock'n'roll,rock'n'roll
129,130,female vocalists,female vocalists,female vocalist,female vocalist
130,131,girl groups,girl groups,girl group,girl group


In [800]:
md("""## Chapter 18 - Preserve review provenance and flag reasons

The review produced two different kinds of notes: explanations of a decision (why `blue` stayed separate from `blues`) and warnings that a tag is personal, subjective or ambiguous. We keep them separate and attached to the tags, so a tag can carry a note without being flagged. Later modules and the final report can then always show why a tag was treated the way it was, instead of us having to remember.""")

## Chapter 18 - Preserve review provenance and flag reasons

The review produced two different kinds of notes: explanations of a decision (why `blue` stayed separate from `blues`) and warnings that a tag is personal, subjective or ambiguous. We keep them separate and attached to the tags, so a tag can carry a note without being flagged. Later modules and the final report can then always show why a tag was treated the way it was, instead of us having to remember.

In [801]:
review_notes_by_key = {}
review_numbers_by_key = {}
manual_flag_notes_by_key = {}

for _, row in APPROVED_PLURAL_REVIEW.iterrows():
    if row["decision"] == "Merge":
        affected_keys = [make_tag_key(row["merge_into"])]
    else:
        affected_keys = [
            make_tag_key(row["singular_tag"]),
            make_tag_key(row["plural_tag"]),
        ]

    affected_keys = [
        equivalence_key_map.get(key, key)
        for key in affected_keys
    ]

    for key in affected_keys:
        review_numbers_by_key.setdefault(key, []).append(int(row["review_no"]))

        if row["reviewer_notes"]:
            review_notes_by_key.setdefault(key, []).append(row["reviewer_notes"])

        if int(row["review_no"]) in FLAGGED_REVIEW_NUMBERS:
            manual_flag_notes_by_key.setdefault(key, []).append(
                row["reviewer_notes"] or "Flagged during manual review."
            )

tag_mapping["review_numbers"] = tag_mapping["canonical_key"].map(
    lambda key: " | ".join(map(str, sorted(set(review_numbers_by_key.get(key, [])))))
)
tag_mapping["manual_review_notes"] = tag_mapping["canonical_key"].map(
    lambda key: " | ".join(dict.fromkeys(review_notes_by_key.get(key, [])))
)
tag_mapping["manual_review_flag"] = tag_mapping["canonical_key"].isin(
    manual_flag_notes_by_key
)

print(
    "Canonical tag keys explicitly flagged by the 73 reviewed pairs:",
    tag_mapping.loc[tag_mapping["manual_review_flag"], "canonical_key"].nunique(),
)

Canonical tag keys explicitly flagged by the 73 reviewed pairs: 85


In [802]:
md("""## Chapter 19 - Apply canonical tags to valid events

Each valid source event is joined to the complete tag mapping.

No tag assignment is silently lost:

- all valid tag IDs receive a canonical key;
- orphan artist rows remain preserved separately;
- the event-level table retains the source tag ID, raw tag, dates, and timestamps.""")

## Chapter 19 - Apply canonical tags to valid events

Each valid source event is joined to the complete tag mapping.

No tag assignment is silently lost:

- all valid tag IDs receive a canonical key;
- orphan artist rows remain preserved separately;
- the event-level table retains the source tag ID, raw tag, dates, and timestamps.

In [803]:
tag_mapping_columns = [
    "tagID",
    "raw_tag",
    "tag_key",
    "canonical_key",
    "canonical_tag",
]

user_taggedartists_events_clean = (
    valid_tag_events
    .merge(
        tag_mapping[tag_mapping_columns],
        on="tagID",
        how="left",
        validate="many_to_one",
    )
)

assert user_taggedartists_events_clean["canonical_key"].notna().all()
assert len(user_taggedartists_events_clean) == len(valid_tag_events)

user_taggedartists_events_clean["timestamp_utc"] = (
    user_taggedartists_events_clean["timestamp_utc_dt"]
    .dt.strftime("%Y-%m-%dT%H:%M:%S%z")
)
user_taggedartists_events_clean["timestamp_madrid"] = (
    user_taggedartists_events_clean["timestamp_madrid_dt"]
    .dt.strftime("%Y-%m-%dT%H:%M:%S%z")
)

user_taggedartists_events_clean = user_taggedartists_events_clean[
    [
        "userID",
        "artistID",
        "source_artistID",
        "tagID",
        "raw_tag",
        "canonical_key",
        "canonical_tag",
        "day",
        "month",
        "year",
        "timestamp_ms",
        "timestamp_utc",
        "timestamp_madrid",
        "is_timestamp_anomaly",
    ]
].sort_values(["userID", "artistID", "canonical_key", "timestamp_ms"]).reset_index(drop=True)

display(user_taggedartists_events_clean.head())

,userID,artistID,source_artistID,tagID,raw_tag,canonical_key,canonical_tag,day,month,year,timestamp_ms,timestamp_utc,timestamp_madrid,is_timestamp_anomaly
0,2,52,52,13,chillout,chill out,chill out,1,4,2009,1238536800000,2009-03-31T22:00:00+0000,2009-04-01T00:00:00+0200,False
1,2,52,52,15,downtempo,downtempo,downtempo,1,4,2009,1238536800000,2009-03-31T22:00:00+0000,2009-04-01T00:00:00+0200,False
2,2,52,52,18,electronic,electronic,electronic,1,4,2009,1238536800000,2009-03-31T22:00:00+0000,2009-04-01T00:00:00+0200,False
3,2,52,52,41,female vovalists,female vovalists,female vovalists,1,4,2009,1238536800000,2009-03-31T22:00:00+0000,2009-04-01T00:00:00+0200,False
4,2,52,52,21,trip-hop,trip hop,trip hop,1,4,2009,1238536800000,2009-03-31T22:00:00+0000,2009-04-01T00:00:00+0200,False


In [804]:
md("""## Chapter 20 - Collapse post-merge duplicate tag votes

The raw files contain no exact duplicate rows. Duplicates can nevertheless be created after:

- artist IDs are merged;
- separator variants are normalized;
- approved singular/plural pairs are merged;
- approved spelling equivalents are merged.

For modelling and community profiles, one person should contribute only one vote to one artist-tag concept.

The cleaned vote table therefore contains one row per:

```text
(userID, artistID, canonical_key)
```

Instead of discarding timing evidence, it retains:

- earliest and latest timestamp;
- number of source assignment rows;
- source tag IDs;
- whether any contributing event has an anomalous timestamp.""")

## Chapter 20 - Collapse post-merge duplicate tag votes

The raw files contain no exact duplicate rows. Duplicates can nevertheless be created after:

- artist IDs are merged;
- separator variants are normalized;
- approved singular/plural pairs are merged;
- approved spelling equivalents are merged.

For modelling and community profiles, one person should contribute only one vote to one artist-tag concept.

The cleaned vote table therefore contains one row per:

```text
(userID, artistID, canonical_key)
```

Instead of discarding timing evidence, it retains:

- earliest and latest timestamp;
- number of source assignment rows;
- source tag IDs;
- whether any contributing event has an anomalous timestamp.

In [805]:
user_taggedartists_clean = (
    user_taggedartists_events_clean
    .groupby(
        ["userID", "artistID", "canonical_key", "canonical_tag"],
        as_index=False,
    )
    .agg(
        first_timestamp_ms=("timestamp_ms", "min"),
        last_timestamp_ms=("timestamp_ms", "max"),
        source_assignment_count=("tagID", "size"),
        source_tag_ids=(
            "tagID",
            lambda values: " | ".join(map(str, sorted(set(values)))),
        ),
        is_timestamp_anomaly=("is_timestamp_anomaly", "max"),
    )
    .sort_values(["userID", "artistID", "canonical_key"])
    .reset_index(drop=True)
)

assert not user_taggedartists_clean.duplicated(
    ["userID", "artistID", "canonical_key"]
).any()

post_merge_duplicates_removed = (
    len(user_taggedartists_events_clean)
    - len(user_taggedartists_clean)
)

assignment_reconciliation = pd.DataFrame([
    {"measure": "valid event rows", "value": len(user_taggedartists_events_clean)},
    {"measure": "unique user-artist-canonical-tag votes", "value": len(user_taggedartists_clean)},
    {"measure": "post-merge duplicate votes consolidated", "value": post_merge_duplicates_removed},
])
display(assignment_reconciliation)

,measure,value
0,valid event rows,184941
1,unique user-artist-canonical-tag votes,182133
2,post-merge duplicate votes consolidated,2808


In [806]:
md("""## Chapter 21 - Build the canonical tag vocabulary

The canonical vocabulary records both usage and provenance:

- distinct users;
- distinct artists;
- cleaned vote rows;
- original source-assignment count;
- number of raw tag IDs and raw variants;
- manual-review notes and flags.

This table is the authoritative lookup used by later text and recommender notebooks.""")

## Chapter 21 - Build the canonical tag vocabulary

The canonical vocabulary records both usage and provenance:

- distinct users;
- distinct artists;
- cleaned vote rows;
- original source-assignment count;
- number of raw tag IDs and raw variants;
- manual-review notes and flags.

This table is the authoritative lookup used by later text and recommender notebooks.

In [807]:
tag_statistics = (
    user_taggedartists_clean
    .groupby(["canonical_key", "canonical_tag"], as_index=False)
    .agg(
        distinct_users=("userID", "nunique"),
        distinct_artists=("artistID", "nunique"),
        vote_rows=("userID", "size"),
        source_assignments=("source_assignment_count", "sum"),
    )
)

tag_variant_statistics = (
    tag_mapping.groupby(["canonical_key", "canonical_tag"], as_index=False)
    .agg(
        raw_tag_id_count=("tagID", "nunique"),
        raw_variant_count=("raw_tag", "nunique"),
        raw_variants=(
            "raw_tag",
            lambda values: " | ".join(sorted(set(map(str, values)))),
        ),
        review_numbers=(
            "review_numbers",
            lambda values: " | ".join(
                sorted(
                    {
                        number
                        for value in values
                        for number in str(value).split(" | ")
                        if number
                    },
                    key=lambda item: int(item),
                )
            ),
        ),
        manual_review_notes=(
            "manual_review_notes",
            lambda values: " | ".join(
                dict.fromkeys(
                    note
                    for value in values
                    for note in [str(value)]
                    if note
                )
            ),
        ),
        manual_review_flag=("manual_review_flag", "max"),
    )
)

tags_clean = (
    tag_statistics
    .merge(
        tag_variant_statistics,
        on=["canonical_key", "canonical_tag"],
        how="left",
        validate="one_to_one",
    )
)

assert tags_clean["canonical_key"].is_unique
assert tags_clean["canonical_tag"].notna().all()

display(
    tags_clean.sort_values(
        ["distinct_users", "source_assignments"],
        ascending=False,
    ).head(15)
)

,canonical_key,canonical_tag,distinct_users,distinct_artists,vote_rows,source_assignments,raw_tag_id_count,raw_variant_count,raw_variants,review_numbers,manual_review_notes,manual_review_flag
7104,rock,rock,671,2248,7459,7459,1,1,rock,,,False
6482,pop,pop,585,1725,5401,5401,1,1,pop,,,False
2549,electronic,electronic,568,1933,5543,6528,8,8,elecronica | electro | electronic | electronica | elertronic | eletro | eletronic | eletronica,198 | 199 | 200 | 201 | 202 | 299 | 300,,False
388,alternative,alternative,539,1724,5234,5240,7,7,alt | alternatif | alternatif muzik | alternativ | alternative | alternativee | alternativo,164 | 165 | 166 | 236 | 237 | 238,,False
2906,female vocalist,female vocalist,479,1468,4652,4878,7,7,female vocalist | female vocalists | female vocals | female voice | female-vocalists | vocal femenino | vocalista fe...,1 | 262 | 263 | 276 | 277,,False
4153,indie,indie,450,1505,4422,4422,1,1,indie,,,False
404,alternative rock,alternative rock,374,863,2619,2622,3,3,alt rock | alt-rock | alternative rock,239,,False
1997,dance,dance,369,931,2726,2726,1,1,dance,,,False
4171,indie rock,indie rock,267,761,2075,2077,4,4,indie rock | indie-rock | indie/rock | indierock,267,,False
40,1980s,1980s,263,812,2853,2950,9,9,'80s | 1980 songs | 1980's | 1980s | 80 | 80's | 80's music | 80s | anos 80,144 | 145 | 146 | 160 | 161 | 222 | 229 | 231 | 242,,False


In [808]:
md("""## Chapter 22 - Add strict music-content eligibility

All cleaned tags are preserved. The `use_for_content_model` field controls only whether a tag is eligible for the later primary TF-IDF model.

The strict vocabulary excludes concepts that clearly describe:

- personal preference or user behaviour;
- subjective praise;
- generic or low-information categories;
- meta or organisational labels;
- artist-name/self-referential labels;
- purely biographical facts;
- cultural status rather than musical content;
- unverifiable identity claims;
- manually reviewed ambiguous or non-musical concepts.

Flagged tags may remain eligible when they clearly describe musical mood, theme, or media context. Examples retained despite flags include `bad day`, `rainy day`, `hymns`, `human condition`, `cowboy`, `jihad song`, and `tv show`.

The automatic patterns are deliberately narrow. Multiword tags exactly matching an artist name are exempt from personal or subjective prefix rules, preventing names such as `my chemical romance` from being incorrectly excluded. Every automatic decision is exported for audit.

The word lists below were not copied from a ready-made stop list. We built them by reading the most-used tags and our own review notes, and writing down the ones that describe the listener rather than the music, like `favorite` or `awesome`. These tags are not deleted, only marked as ineligible for the Module 3 search engine, because a tag like `my favorite` says nothing about what an artist sounds like and would only add noise to the recommendations in Module 4.""")

## Chapter 22 - Add strict music-content eligibility

All cleaned tags are preserved. The `use_for_content_model` field controls only whether a tag is eligible for the later primary TF-IDF model.

The strict vocabulary excludes concepts that clearly describe:

- personal preference or user behaviour;
- subjective praise;
- generic or low-information categories;
- meta or organisational labels;
- artist-name/self-referential labels;
- purely biographical facts;
- cultural status rather than musical content;
- unverifiable identity claims;
- manually reviewed ambiguous or non-musical concepts.

Flagged tags may remain eligible when they clearly describe musical mood, theme, or media context. Examples retained despite flags include `bad day`, `rainy day`, `hymns`, `human condition`, `cowboy`, `jihad song`, and `tv show`.

The automatic patterns are deliberately narrow. Multiword tags exactly matching an artist name are exempt from personal or subjective prefix rules, preventing names such as `my chemical romance` from being incorrectly excluded. Every automatic decision is exported for audit.

The word lists below were not copied from a ready-made stop list. We built them by reading the most-used tags and our own review notes, and writing down the ones that describe the listener rather than the music, like `favorite` or `awesome`. These tags are not deleted, only marked as ineligible for the Module 3 search engine, because a tag like `my favorite` says nothing about what an artist sounds like and would only add noise to the recommendations in Module 4.

In [809]:
ALLOWED_FLAGGED_CONTENT_TAGS = {
    make_tag_key(value)
    for value in [
        "bad day",
        "rainy day",
        "jihad song",
        "cowboy",
        "hymns",
        "human condition",
        "tv show",
    ]
}

PERSONAL_PATTERNS = [
    r"\b(?:my|mine|me|i|we|our)\b",
    r"\bseen live\b",
    r"\bheard live\b",
    r"\balbums? i own\b",
    r"\bwant to see\b",
    r"\bto buy\b",
    r"\bwishlist\b",
    r"\bnew discovery\b",
    r"\bguilty pleasure\b",
    r"\bfavou?rites?\b",
    r"\bfavs?\b",
    r"\bfaves?\b",
    r"\bplay yet\b",
    r"\bcheck out\b",
    r"\bto check\b",
    r"\blisten to\b",
    r"\bdownload\b",
    r"\bowned\b",
    r"\bcollection\b",
    r"\btags\b",
    r"\btag your songs properly\b",
    r"\byou\b",
    r"\bwhoah yeah\b",
    r"\bwho is goth lol\b",
    r"\brandom\b",
    r"\brandom songs\b",
    r"\b-pearl fashion music\b",
    r"\bbands that start with the\b",
]

SUBJECTIVE_EXACT = {
    make_tag_key(value)
    for value in [
        "amazing",
        "awesome",
        "awesomecore",
        "awesomeness",
        "awesomesauce",
        "awsome",
        "beautiful",
        "best",
        "brilliant",
        "cool",
        "epic",
        "excellent",
        "fantastic",
        "fun",
        "genius",
        "good",
        "gorgeous",
        "great",
        "incredible",
        "legend",
        "love",
        "lovely",
        "masterpiece",
        "nice",
        "perfect",
        "sexy",
        "sweet",
        "wonderful",
    ]
}

SUBJECTIVE_PATTERNS = [
    r"^(?:great|best|awesome|perfect|beautiful|heavenly|sexy)\b",
    r"^(?:legend|masterpiece|guitar god|genius)\b",
    r"\blove at first listen\b",
    r"^love (?:it|him|her|this|these|them)\b",
    r"^reasons to love\b",
]

GENERIC_EXACT = {
    make_tag_key(value)
    for value in ["song", "track", "tune", "band", "record", "music", "artist"]
}
META_EXACT = {make_tag_key(value) for value in ["as tag", "blog"]}
SELF_REFERENTIAL_EXACT = {
    make_tag_key(value)
    for value in ["fractured visions", "visions"]
}
BIOGRAPHICAL_EXACT = {make_tag_key("actor")}
CULTURAL_STATUS_EXACT = {make_tag_key("gay icon")}
IDENTITY_RELATED_EXACT = {make_tag_key("lesbian")}

artist_name_tag_keys = {
    make_tag_key(name)
    for name in artists_clean["source_name_repaired"].astype(str)
}

def automatic_content_exclusion_reason(canonical_key):
    if canonical_key in GENERIC_EXACT:
        return "generic or low-information"
    if canonical_key in META_EXACT:
        return "meta or organisational"
    if canonical_key in SELF_REFERENTIAL_EXACT:
        return "artist-name or self-referential"
    if canonical_key in BIOGRAPHICAL_EXACT:
        return "biographical and non-musical"
    if canonical_key in CULTURAL_STATUS_EXACT:
        return "cultural status rather than musical content"
    if canonical_key in IDENTITY_RELATED_EXACT:
        return "identity-related and not verified musical content"
    if canonical_key in SUBJECTIVE_EXACT:
        return "subjective evaluation"

    # Avoid false positives for multiword artist names such as
    # "my chemical romance" and "my bloody valentine".
    if " " in canonical_key and canonical_key in artist_name_tag_keys:
        return ""

    if any(re.search(pattern, canonical_key) for pattern in PERSONAL_PATTERNS):
        return "personal preference or user behaviour"

    if any(re.search(pattern, canonical_key) for pattern in SUBJECTIVE_PATTERNS):
        return "subjective evaluation"

    return ""

tags_clean["automatic_exclusion_reason"] = (
    tags_clean["canonical_key"].map(automatic_content_exclusion_reason)
)

manual_flag_reason_lookup = {
    key: " | ".join(dict.fromkeys(notes))
    for key, notes in manual_flag_notes_by_key.items()
}

tags_clean["manual_flag_reason"] = tags_clean["canonical_key"].map(
    manual_flag_reason_lookup
).fillna("")

tags_clean["is_flagged"] = (
    tags_clean["manual_review_flag"]
    | tags_clean["automatic_exclusion_reason"].ne("")
)

tags_clean["semantic_content_eligible"] = True

manual_disallowed = (
    tags_clean["manual_review_flag"]
    & ~tags_clean["canonical_key"].isin(ALLOWED_FLAGGED_CONTENT_TAGS)
)
tags_clean.loc[manual_disallowed, "semantic_content_eligible"] = False

automatic_disallowed = tags_clean["automatic_exclusion_reason"].ne("")
tags_clean.loc[automatic_disallowed, "semantic_content_eligible"] = False

def combine_semantic_reasons(row):
    reasons = []
    if row["manual_review_flag"] and row["canonical_key"] not in ALLOWED_FLAGGED_CONTENT_TAGS:
        reasons.append(row["manual_flag_reason"] or "excluded by manual review")
    if row["automatic_exclusion_reason"]:
        reasons.append(row["automatic_exclusion_reason"])
    return " | ".join(dict.fromkeys(reasons))

tags_clean["semantic_exclusion_reason"] = tags_clean.apply(
    combine_semantic_reasons,
    axis=1,
)

display(
    tags_clean.loc[
        tags_clean["is_flagged"],
        [
            "canonical_tag",
            "manual_review_flag",
            "automatic_exclusion_reason",
            "semantic_content_eligible",
            "semantic_exclusion_reason",
        ],
    ].head(20)
)

,canonical_tag,manual_review_flag,automatic_exclusion_reason,semantic_content_eligible,semantic_exclusion_reason
0,0 play yet,False,personal preference or user behaviour,False,personal preference or user behaviour
120,60s favorites,False,personal preference or user behaviour,False,personal preference or user behaviour
152,80s i like,False,personal preference or user behaviour,False,personal preference or user behaviour
206,a piece of me,False,personal preference or user behaviour,False,personal preference or user behaviour
261,actor,True,biographical and non-musical,False,Flagged: biographical/non-musical. | biographical and non-musical
330,album favourite,False,personal preference or user behaviour,False,personal preference or user behaviour
333,albums i own,False,personal preference or user behaviour,False,personal preference or user behaviour
361,all i ever wanted,False,personal preference or user behaviour,False,personal preference or user behaviour
364,all the people you love in a river of blood,False,personal preference or user behaviour,False,personal preference or user behaviour
366,all time faves,False,personal preference or user behaviour,False,personal preference or user behaviour


In [810]:
md("""## Chapter 23 - Apply the rare-tag document-frequency rule

A tag that appears on only one artist cannot connect that artist to any other, so it is useless for measuring similarity. It stays in the vocabulary file but is marked ineligible for the Module 3 search matrix. We set no upper limit: broad tags like `rock` and `pop` stay in, because TF-IDF automatically gives very common tags less influence, so removing them by hand is unnecessary.""")

## Chapter 23 - Apply the rare-tag document-frequency rule

A tag that appears on only one artist cannot connect that artist to any other, so it is useless for measuring similarity. It stays in the vocabulary file but is marked ineligible for the Module 3 search matrix. We set no upper limit: broad tags like `rock` and `pop` stay in, because TF-IDF automatically gives very common tags less influence, so removing them by hand is unnecessary.

In [811]:
MIN_TAG_ARTISTS = 2
MAX_TAG_ARTIST_SHARE = None

tags_clean["single_user_tag"] = tags_clean["distinct_users"] == 1

tags_clean["meets_min_artist_threshold"] = (
    tags_clean["distinct_artists"] >= MIN_TAG_ARTISTS
)

tags_clean["use_for_content_model"] = (
    tags_clean["semantic_content_eligible"]
    & tags_clean["meets_min_artist_threshold"]
)

tags_clean["content_model_exclusion_reason"] = np.select(
    [
        ~tags_clean["semantic_content_eligible"],
        ~tags_clean["meets_min_artist_threshold"],
    ],
    [
        tags_clean["semantic_exclusion_reason"],
        f"used on fewer than {MIN_TAG_ARTISTS} distinct artists",
    ],
    default="",
)

vocabulary_summary = pd.DataFrame([
    {"measure": "raw tag IDs", "value": len(tags_raw)},
    {"measure": "basic normalized keys", "value": tag_mapping["tag_key"].nunique()},
    {"measure": "final canonical keys in lookup", "value": tag_mapping["canonical_key"].nunique()},
    {"measure": "used canonical tags", "value": len(tags_clean)},
    {"measure": "used by exactly one user", "value": int(tags_clean["single_user_tag"].sum())},
    {"measure": "used on exactly one artist", "value": int((tags_clean["distinct_artists"] == 1).sum())},
    {"measure": "semantically eligible tags", "value": int(tags_clean["semantic_content_eligible"].sum())},
    {"measure": "primary content-model tags", "value": int(tags_clean["use_for_content_model"].sum())},
    {"measure": "hard maximum-frequency filter", "value": "None - rely on IDF"},
])
display(vocabulary_summary)

# Carry eligibility directly into both clean assignment tables.
tag_assignment_attributes = tags_clean[
    [
        "canonical_key",
        "is_flagged",
        "semantic_content_eligible",
        "meets_min_artist_threshold",
        "use_for_content_model",
        "content_model_exclusion_reason",
    ]
].rename(columns={"is_flagged": "tag_is_flagged"})

user_taggedartists_clean = user_taggedartists_clean.merge(
    tag_assignment_attributes,
    on="canonical_key",
    how="left",
    validate="many_to_one",
)

user_taggedartists_events_clean = user_taggedartists_events_clean.merge(
    tag_assignment_attributes,
    on="canonical_key",
    how="left",
    validate="many_to_one",
)

assert user_taggedartists_clean["use_for_content_model"].notna().all()
assert user_taggedartists_events_clean["use_for_content_model"].notna().all()

display(
    user_taggedartists_clean[
        [
            "userID",
            "artistID",
            "canonical_tag",
            "tag_is_flagged",
            "use_for_content_model",
            "content_model_exclusion_reason",
        ]
    ].head()
)

,measure,value
0,raw tag IDs,11946
1,basic normalized keys,11801
2,final canonical keys in lookup,11486
3,used canonical tags,9297
4,used by exactly one user,7337
5,used on exactly one artist,5392
6,semantically eligible tags,8511
7,primary content-model tags,3500
8,hard maximum-frequency filter,None - rely on IDF


,userID,artistID,canonical_tag,tag_is_flagged,use_for_content_model,content_model_exclusion_reason
0,2,52,chill out,False,True,
1,2,52,downtempo,False,True,
2,2,52,electronic,False,True,
3,2,52,female vovalists,False,False,used on fewer than 2 distinct artists
4,2,52,trip hop,False,True,


In [812]:
md("""## Chapter 24 - Prepare artist-tag support counts

Module 3 will need to know how many independent people support each artist-tag pair: five users calling an artist `jazz` is stronger evidence than one. We export the plain count and leave the mathematical weighting to Module 3 itself. That way this notebook only records what was observed, and every modelling choice stays visible in the module that actually makes it.""")

## Chapter 24 - Prepare artist-tag support counts

Module 3 will need to know how many independent people support each artist-tag pair: five users calling an artist `jazz` is stronger evidence than one. We export the plain count and leave the mathematical weighting to Module 3 itself. That way this notebook only records what was observed, and every modelling choice stays visible in the module that actually makes it.

In [813]:
content_eligibility_lookup = tags_clean.set_index("canonical_key")[
    "use_for_content_model"
]

artist_tag_support_clean = (
    user_taggedartists_clean
    .groupby(["artistID", "canonical_key", "canonical_tag"], as_index=False)
    .agg(
        distinct_user_count=("userID", "nunique"),
        vote_rows=("userID", "size"),
        first_timestamp_ms=("first_timestamp_ms", "min"),
        last_timestamp_ms=("last_timestamp_ms", "max"),
    )
)

artist_tag_support_clean["use_for_content_model"] = (
    artist_tag_support_clean["canonical_key"]
    .map(content_eligibility_lookup)
    .fillna(False)
    .astype(bool)
)

eligible_profile_counts = (
    artist_tag_support_clean[
        artist_tag_support_clean["use_for_content_model"]
    ]
    .groupby("artistID")["canonical_key"]
    .nunique()
)

artists_clean["eligible_content_tag_count"] = (
    artists_clean["artistID"]
    .map(eligible_profile_counts)
    .fillna(0)
    .astype("int64")
)
artists_clean["has_content_profile"] = (
    artists_clean["eligible_content_tag_count"] > 0
)

assert (
    artist_tag_support_clean.loc[
        artist_tag_support_clean["use_for_content_model"],
        "canonical_key",
    ].isin(
        tags_clean.loc[
            tags_clean["use_for_content_model"],
            "canonical_key",
        ]
    ).all()
)

artist_profile_summary = pd.DataFrame([
    {"measure": "artists in clean catalogue", "value": len(artists_clean)},
    {"measure": "artists with an eligible content profile", "value": int(artists_clean["has_content_profile"].sum())},
    {"measure": "artists retained but excluded from pure content similarity", "value": int((~artists_clean["has_content_profile"]).sum())},
])
display(artist_profile_summary)

,measure,value
0,artists in clean catalogue,17619
1,artists with an eligible content profile,11839
2,artists retained but excluded from pure content similarity,5780


In [814]:
md("""## Chapter 25 - Final integrity and reconciliation checks

The pipeline now validates the complete relational output.

The assertions verify:

- expected source and clean row counts;
- artist-ID mapping completeness;
- listening-weight preservation;
- friendship symmetry and undirected uniqueness;
- complete tag-ID mapping;
- no orphan artist IDs in analytical tag tables;
- one vote per user-artist-canonical-tag;
- content eligibility only when document frequency is at least two artists;
- all five timestamp anomalies preserved and flagged.

If any assertion fails, exports should not be trusted.""")

## Chapter 25 - Final integrity and reconciliation checks

The pipeline now validates the complete relational output.

The assertions verify:

- expected source and clean row counts;
- artist-ID mapping completeness;
- listening-weight preservation;
- friendship symmetry and undirected uniqueness;
- complete tag-ID mapping;
- no orphan artist IDs in analytical tag tables;
- one vote per user-artist-canonical-tag;
- content eligibility only when document frequency is at least two artists;
- all five timestamp anomalies preserved and flagged.

If any assertion fails, exports should not be trusted.

In [815]:
final_artist_ids = set(artists_clean["artistID"])
final_tag_keys = set(tags_clean["canonical_key"])

assert len(artists_clean) == 17_619
assert len(artist_id_mapping) == 17_632
assert artist_id_mapping["source_artistID"].is_unique
assert artist_id_mapping["final_artistID"].isin(final_artist_ids).all()

assert len(user_artists_clean) == 92_829
assert set(user_artists_clean["artistID"]).issubset(final_artist_ids)
assert int(user_artists_clean["weight"].sum()) == int(user_artists_raw["weight"].sum())

assert len(user_friends_directed_clean) == 25_434
assert len(user_friends_undirected_clean) == 12_717

assert len(orphan_assignments_removed) == 1_538
assert orphan_assignments_removed["source_artistID"].nunique() == 390
assert len(user_taggedartists_events_clean) == 184_941
assert set(user_taggedartists_events_clean["artistID"]).issubset(final_artist_ids)
assert set(user_taggedartists_clean["canonical_key"]).issubset(final_tag_keys)
assert user_taggedartists_clean["use_for_content_model"].notna().all()
assert user_taggedartists_events_clean["use_for_content_model"].notna().all()
assert not user_taggedartists_clean.duplicated(
    ["userID", "artistID", "canonical_key"]
).any()

assert (
    tags_clean.loc[
        tags_clean["use_for_content_model"],
        "distinct_artists",
    ] >= MIN_TAG_ARTISTS
).all()

assert int(user_taggedartists_events_clean["is_timestamp_anomaly"].sum()) == 5

final_validation = pd.DataFrame([
    {"check": "clean artist IDs unique", "passed": artists_clean["artistID"].is_unique},
    {"check": "all mapped artists resolve", "passed": artist_id_mapping["final_artistID"].isin(final_artist_ids).all()},
    {"check": "listening total preserved", "passed": int(user_artists_clean["weight"].sum()) == int(user_artists_raw["weight"].sum())},
    {"check": "undirected friendship edges unique", "passed": not user_friends_undirected_clean.duplicated(["userID_a", "userID_b"]).any()},
    {"check": "all valid tag artists resolve", "passed": set(user_taggedartists_events_clean["artistID"]).issubset(final_artist_ids)},
    {"check": "clean tag votes unique", "passed": not user_taggedartists_clean.duplicated(["userID", "artistID", "canonical_key"]).any()},
    {"check": "all eligible tags meet min document frequency", "passed": (tags_clean.loc[tags_clean["use_for_content_model"], "distinct_artists"] >= MIN_TAG_ARTISTS).all()},
    {"check": "all timestamp anomalies retained", "passed": int(user_taggedartists_events_clean["is_timestamp_anomaly"].sum()) == 5},
])

assert final_validation["passed"].all()
display(final_validation)

,check,passed
0,clean artist IDs unique,True
1,all mapped artists resolve,True
2,listening total preserved,True
3,undirected friendship edges unique,True
4,all valid tag artists resolve,True
5,clean tag votes unique,True
6,all eligible tags meet min document frequency,True
7,all timestamp anomalies retained,True


In [816]:
md("""## Chapter 26 - Build the cleaning summary

The summary records the most important before-and-after measures in one machine-readable table. Later report notebooks can load this file directly rather than copying numbers manually.""")

## Chapter 26 - Build the cleaning summary

The summary records the most important before-and-after measures in one machine-readable table. Later report notebooks can load this file directly rather than copying numbers manually.

In [817]:
cleaning_summary = pd.DataFrame([
    {"section": "source", "measure": "users", "value": len(users_clean)},
    {"section": "source", "measure": "raw artists", "value": len(artists_raw)},
    {"section": "artists", "measure": "duplicate groups reviewed", "value": len(duplicate_artist_groups)},
    {"section": "artists", "measure": "absorbed artist IDs", "value": len(absorbed_artist_ids)},
    {"section": "artists", "measure": "clean artists", "value": len(artists_clean)},
    {"section": "artists", "measure": "targeted encoding repairs", "value": int(artists_clean["source_name_raw"].ne(artists_clean["source_name_repaired"]).sum())},
    {"section": "listening", "measure": "raw rows", "value": len(user_artists_raw)},
    {"section": "listening", "measure": "clean rows", "value": len(user_artists_clean)},
    {"section": "listening", "measure": "total weight before", "value": total_listening_weight_before},
    {"section": "listening", "measure": "total weight after", "value": total_listening_weight_after},
    {"section": "friendships", "measure": "directed rows", "value": len(user_friends_directed_clean)},
    {"section": "friendships", "measure": "undirected edges", "value": len(user_friends_undirected_clean)},
    {"section": "tags", "measure": "raw tag IDs", "value": len(tags_raw)},
    {"section": "tags", "measure": "basic normalized keys", "value": tag_mapping["tag_key"].nunique()},
    {"section": "tags", "measure": "final canonical keys in lookup", "value": tag_mapping["canonical_key"].nunique()},
    {"section": "tags", "measure": "used canonical tags", "value": len(tags_clean)},
    {"section": "tags", "measure": "approved review pairs merged", "value": int((APPROVED_PLURAL_REVIEW["decision"] == "Merge").sum())},
    {"section": "tags", "measure": "approved review pairs kept separate", "value": int((APPROVED_PLURAL_REVIEW["decision"] == "Keep separate").sum())},
    {"section": "tags", "measure": "approved review pairs flagged", "value": len(FLAGGED_REVIEW_NUMBERS)},
    {"section": "tags", "measure": "content-model eligible tags", "value": int(tags_clean["use_for_content_model"].sum())},
    {"section": "tag assignments", "measure": "raw events", "value": len(tag_events)},
    {"section": "tag assignments", "measure": "orphan events removed from analysis", "value": len(orphan_assignments_removed)},
    {"section": "tag assignments", "measure": "valid event rows", "value": len(user_taggedartists_events_clean)},
    {"section": "tag assignments", "measure": "unique cleaned votes", "value": len(user_taggedartists_clean)},
    {"section": "timestamps", "measure": "pre-2005 anomalies retained", "value": int(user_taggedartists_events_clean["is_timestamp_anomaly"].sum())},
    {"section": "content profile", "measure": "artists with eligible profile", "value": int(artists_clean["has_content_profile"].sum())},
])

display(cleaning_summary)

,section,measure,value
0,source,users,1892
1,source,raw artists,17632
2,artists,duplicate groups reviewed,12
3,artists,absorbed artist IDs,13
4,artists,clean artists,17619
5,artists,targeted encoding repairs,1
6,listening,raw rows,92834
7,listening,clean rows,92829
8,listening,total weight before,69183975
9,listening,total weight after,69183975


In [818]:
md("""## Chapter 27 - Export the cleaned relational tables

CSV is used because it is transparent, portable, version-control friendly, and directly loadable by pandas, NetworkX, and scikit-learn.

The output folder contains:

| File | Purpose |
|---|---|
| `users_clean.csv` | derived user universe and presence flags |
| `artists_clean.csv` | deduplicated and canonical artist lookup |
| `artist_id_mapping.csv` | every raw artist ID mapped to its final ID |
| `artist_merge_audit.csv` | reviewed duplicate groups and survivor evidence |
| `user_artists_clean.csv` | remapped implicit-feedback table |
| `user_friends_directed_clean.csv` | source-equivalent directed edges |
| `user_friends_undirected_clean.csv` | one row per social edge |
| `approved_tag_review_decisions.csv` | the 140 approved manual decisions |
| `tag_mapping.csv` | every raw tag ID mapped to a canonical key |
| `tags_clean.csv` | canonical vocabulary, usage, flags, and eligibility |
| `user_taggedartists_events_clean.csv` | every valid source tagging event |
| `user_taggedartists_clean.csv` | one cleaned vote per user-artist-tag |
| `artist_tag_support_clean.csv` | distinct-user support for artist-tag pairs |
| `orphan_assignments_removed.csv` | excluded unknown-artist evidence |
| `timestamp_anomalies.csv` | the five retained anomalous dates |
| `cleaning_summary.csv` | before-and-after metrics |""")

## Chapter 27 - Export the cleaned relational tables

CSV is used because it is transparent, portable, version-control friendly, and directly loadable by pandas, NetworkX, and scikit-learn.

The output folder contains:

| File | Purpose |
|---|---|
| `users_clean.csv` | derived user universe and presence flags |
| `artists_clean.csv` | deduplicated and canonical artist lookup |
| `artist_id_mapping.csv` | every raw artist ID mapped to its final ID |
| `artist_merge_audit.csv` | reviewed duplicate groups and survivor evidence |
| `user_artists_clean.csv` | remapped implicit-feedback table |
| `user_friends_directed_clean.csv` | source-equivalent directed edges |
| `user_friends_undirected_clean.csv` | one row per social edge |
| `approved_tag_review_decisions.csv` | the 140 approved manual decisions |
| `tag_mapping.csv` | every raw tag ID mapped to a canonical key |
| `tags_clean.csv` | canonical vocabulary, usage, flags, and eligibility |
| `user_taggedartists_events_clean.csv` | every valid source tagging event |
| `user_taggedartists_clean.csv` | one cleaned vote per user-artist-tag |
| `artist_tag_support_clean.csv` | distinct-user support for artist-tag pairs |
| `orphan_assignments_removed.csv` | excluded unknown-artist evidence |
| `timestamp_anomalies.csv` | the five retained anomalous dates |
| `cleaning_summary.csv` | before-and-after metrics |

In [819]:
timestamp_anomalies = user_taggedartists_events_clean.loc[
    user_taggedartists_events_clean["is_timestamp_anomaly"]
].copy()

EXPORTS = {
    "users_clean.csv": users_clean,
    "artists_clean.csv": artists_clean,
    "artist_id_mapping.csv": artist_id_mapping,
    "artist_merge_audit.csv": artist_merge_audit,
    "user_artists_clean.csv": user_artists_clean,
    "user_friends_directed_clean.csv": user_friends_directed_clean,
    "user_friends_undirected_clean.csv": user_friends_undirected_clean,
    "approved_tag_review_decisions.csv": APPROVED_PLURAL_REVIEW,
    "tag_mapping.csv": tag_mapping,
    "tags_clean.csv": tags_clean,
    "user_taggedartists_events_clean.csv": user_taggedartists_events_clean,
    "user_taggedartists_clean.csv": user_taggedartists_clean,
    "artist_tag_support_clean.csv": artist_tag_support_clean,
    "orphan_assignments_removed.csv": orphan_assignments_removed,
    "timestamp_anomalies.csv": timestamp_anomalies,
    "cleaning_summary.csv": cleaning_summary,
}

export_manifest_rows = []
for filename, frame in EXPORTS.items():
    output_path = OUTPUT_DIR / filename
    frame.to_csv(output_path, index=False, encoding="utf-8")
    export_manifest_rows.append({
        "filename": filename,
        "rows": len(frame),
        "columns": len(frame.columns),
        "size_bytes": output_path.stat().st_size,
    })

export_manifest = pd.DataFrame(export_manifest_rows)
export_manifest.to_csv(
    OUTPUT_DIR / "export_manifest.csv",
    index=False,
    encoding="utf-8",
)

display(export_manifest)

,filename,rows,columns,size_bytes
0,users_clean.csv,1892,4,36895
1,artists_clean.csv,17619,8,2538151
2,artist_id_mapping.csv,17632,5,782074
3,artist_merge_audit.csv,25,8,2454
4,user_artists_clean.csv,92829,4,1389229
5,user_friends_directed_clean.csv,25434,2,226130
6,user_friends_undirected_clean.csv,12717,2,113075
7,approved_tag_review_decisions.csv,343,8,21891
8,tag_mapping.csv,11946,14,1372396
9,tags_clean.csv,9297,21,1143545
